# PANDA Block — Standalone Notebook
### Peer-Anchored Named Deliberation Architecture | CDDM Dataset

**Reads VIDA R1 output files you upload, then runs R2+R3 named debate.**

Required input files (upload via Kaggle sidebar → Add Input → Upload Dataset):
- `r1_results.csv` — per-agent R1 predictions for all images
- `r1_metrics.csv` — per-agent accuracy metrics from VIDA
- `r1_raw_bank.json` — raw model responses for all images

---
## Workflow
1. Upload VIDA output files → Kaggle sidebar
2. Run **Cell 1** (install)
3. Run **Cell 2** (imports + API keys)
4. Run **Cell 3** (taxonomy + parser)
5. Run **Cell 4** (inference functions)
6. Run **Cell 5** (prompts + voting)
7. Run **Cell 6 — File Debug** → verify files load correctly
8. Run **Cell 7** (agent selection)
9. Run **Cell 8 — Pre-PANDA Test** → send output to verify debate format
10. Run **Cell 9** (full PANDA run)
11. Run **Cells 10–15** (analysis + visualization)
---

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install openai anthropic together pillow pandas matplotlib seaborn tqdm scikit-learn requests -q
print("Dependencies ready")

## Cell 2: Imports & API Keys

In [ ]:
import os, json, base64, random, time, re, math, shutil, traceback
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import requests as _requests
from pathlib import Path
from PIL import Image as PILImage
from tqdm import tqdm
from collections import Counter, defaultdict

import openai
import anthropic
from together import Together

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()

openai_client    = openai.OpenAI(api_key=_secrets.get_secret("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=_secrets.get_secret("ANTHROPIC_API_KEY"))
xai_client       = openai.OpenAI(
    api_key=_secrets.get_secret("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)
_together_compat = openai.OpenAI(
    api_key=_secrets.get_secret("TOGETHER_API_KEY"),
    base_url="https://api.together.xyz/v1",
    timeout=90.0, max_retries=0,
)

print("OpenAI    :", openai_client.base_url)
print("Anthropic : initialized")
print("xAI       :", xai_client.base_url)
print("Together  : initialized")

## Cell 3: Config, Taxonomy & Parser

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_cddm_candidates = [
    Path("/kaggle/input/cddm-dataset/images"),
    Path("/kaggle/input/datasets/maljuboori/cddm-dataset/images"),
]
DATASET_ROOT = None
for _c in _cddm_candidates:
    if _c.exists(): DATASET_ROOT = _c; break
if DATASET_ROOT is None:
    for p in Path("/kaggle/input").rglob("images"):
        if p.is_dir() and any(p.iterdir()): DATASET_ROOT = p; break

OUTPUT_DIR = Path("/kaggle/working/results_panda")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED      = 42
SOFTMAX_TEMP     = 2.0
HEALTHY_LABEL    = "Healthy"
CHECKPOINT_EVERY = 10

# ── Judge / leakage / open-model config (review revisions) ────────────────────
JUDGE_MODEL          = "gpt-5"  # non-participant judge for semantic metrics (reviewer R1.3). Swappable.
MENTION_DATASET_NAME = False    # False = withhold 'CDDM' from debate prompts (reviewer R2-Meth1)
QWEN_MODEL           = "Qwen/Qwen3.5-9B"   # SERVERLESS vision model, light+cheap (reviewer R2-Meth2 lighter setup)
QWEN_FALLBACKS       = [        # ONLY confirmed-serverless Qwen vision models (non-serverless ones 400)
    "Qwen/Qwen3.5-9B",
    "Qwen/Qwen3.5-397B-A17B",
]

random.seed(RANDOM_SEED)

# ── CDDM Taxonomy ─────────────────────────────────────────────────────────────
CDDM_CLASSES = [
    "Apple,Alternaria Blotch","Apple,Black Rot","Apple,Brown Spot",
    "Apple,Cedar Apple Rust","Apple,Frog Eye Leaf Spot","Apple,Grey Spot",
    "Apple,Healthy","Apple,Leaf Rust","Apple,Mosaic Virus",
    "Apple,Powdery Mildew","Apple,Scab",
    "Bell Pepper,Bacterial Spot","Bell Pepper,Healthy",
    "Blueberry,Healthy","Cherry,Healthy","Cherry,Powdery Mildew",
    "Corn,Healthy","Corn,Leaf Rust","Corn,Leaf Spot","Corn,Northern Leaf Blight",
    "Grape,Black Rot","Grape,Esca","Grape,Healthy","Grape,Leaf Blight",
    "Orange,Citrus Greening","Orange,Healthy",
    "Peach,Bacterial Spot","Peach,Healthy",
    "Potato,Early Blight","Potato,Healthy","Potato,Late Blight",
    "Pumpkin,Powdery Mildew","Raspberry,Healthy",
    "Rice,Bacterial Leaf Blight","Rice,Blast","Rice,Brown Spot",
    "Rice,Leaf Blight","Rice,Leaf Smut","Rice,Tungro",
    "Soybean,Healthy","Strawberry,Healthy","Strawberry,Leaf Scorch",
    "Tomato,Bacterial Spot","Tomato,Early Blight","Tomato,Healthy",
    "Tomato,Late Blight","Tomato,Leaf Mold","Tomato,Mosaic Virus",
    "Tomato,Powdery Mildew","Tomato,Septoria Leaf Spot","Tomato,Spider Mites",
    "Tomato,Target Spot","Tomato,Yellow Leaf Curl","Wheat,Healthy",
]
CDDM_CROPS    = sorted(set(c.split(",")[0].strip() for c in CDDM_CLASSES))
CDDM_DISEASES = sorted(set(c.split(",")[1].strip() for c in CDDM_CLASSES))
_CROP_LIST    = ", ".join(CDDM_CROPS)
_DISEASE_LIST = ", ".join(CDDM_DISEASES)

print(f"Taxonomy  : {len(CDDM_CLASSES)} classes | {len(CDDM_CROPS)} crops | {len(CDDM_DISEASES)} diseases")
print(f"Dataset   : {DATASET_ROOT or 'NOT FOUND — add via Kaggle sidebar'}")
print(f"Output    : {OUTPUT_DIR}")

# ── Parser (3-pass: exact → reverse-substring → word-overlap) ─────────────────
def strip_md(text):
    text = re.sub(r"```[\w]*\n?","",text)
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}",r"\1",text)
    return text.strip()

def _match_crop(t):
    t = t.strip().lower()
    all_m = set()
    for c in CDDM_CROPS:
        if c.lower() in t: all_m.add(c)
    for c in CDDM_CROPS:
        if t and t in c.lower(): all_m.add(c)
    if all_m: return sorted(all_m, key=len, reverse=True)[0]
    t_words = set(t.split())
    best, best_score = None, 0
    for c in sorted(CDDM_CROPS, key=len, reverse=True):
        overlap = len(t_words & set(c.lower().split()))
        if overlap > best_score and (overlap >= 2 or overlap == len(t_words)):
            best, best_score = c, overlap
    return best

def _match_disease(t):
    t = t.strip().lower()
    all_m = set()
    for d in CDDM_DISEASES:
        if d.lower() in t: all_m.add(d)
    for d in CDDM_DISEASES:
        if t and t in d.lower(): all_m.add(d)
    if all_m: return sorted(all_m, key=len, reverse=True)[0]
    t_words = set(t.split())
    best, best_score, best_len = None, 0, 0
    for d in sorted(CDDM_DISEASES, key=len, reverse=True):
        overlap = len(t_words & set(d.lower().split()))
        if overlap >= 2 or (overlap >= 1 and overlap == len(t_words)):
            if overlap > best_score or (overlap == best_score and len(d) > best_len):
                best, best_score, best_len = d, overlap, len(d)
    return best

def parse_response(text, agent_name=""):
    result = {"crop":None,"disease":None,"diseased":None,
              "reasoning":"","confidence":None,
              "influenced_by":None,"new_visual_evidence":None,"raw":text}
    if not text or str(text).startswith("ERROR"): return result
    clean=strip_md(str(text)); clean_l=clean.lower()
    lines=clean.split("\n"); in_rsn=False; rsn_lines=[]
    known=["crop category:","disease category:","diseased:","reasoning:",
           "confidence:","influenced by:","new visual evidence:"]
    for line in lines:
        s=line.strip(); sl=s.lower()
        if any(sl.startswith(f) for f in known): in_rsn=False
        if sl.startswith("crop category:") or sl.startswith("crop:"):
            val=s.split(":",1)[1].strip()
            result["crop"]=(next((c for c in CDDM_CROPS if c.lower()==val.lower()),None)
                            or _match_crop(val.lower()))
        elif sl.startswith("disease category:") or sl.startswith("disease:"):
            val=s.split(":",1)[1].strip()
            result["disease"]=(next((d for d in CDDM_DISEASES if d.lower()==val.lower()),None)
                               or _match_disease(val.lower()))
        elif sl.startswith("diseased:"):
            val=s.split(":",1)[1].strip().lower()
            if re.search(r"\byes\b",val): result["diseased"]=1
            elif re.search(r"\bno\b",val): result["diseased"]=0
        elif sl.startswith("reasoning:"):
            result["reasoning"]=s.split(":",1)[1].strip(); in_rsn=True
        elif in_rsn and s: rsn_lines.append(s)
        elif sl.startswith("confidence:"):
            nums=re.findall(r"[0-9]*\.?[0-9]+",s.split(":",1)[1])
            if nums:
                c=float(nums[0]); result["confidence"]=c/100 if c>1 else c
        elif sl.startswith("influenced by:"):
            val=s.split(":",1)[1].strip()
            result["influenced_by"]="None" if val.lower() in (
                "none","n/a","na","no one","nobody","") else val
        elif sl.startswith("new visual evidence:"):
            val=s.split(":",1)[1].strip()
            result["new_visual_evidence"]="NONE" if val.upper() in (
                "NONE","N/A","NA","NOTHING","") else val
    if rsn_lines: result["reasoning"]=(result["reasoning"]+" "+" ".join(rsn_lines)).strip()
    if result["crop"] is None: result["crop"]=_match_crop(clean_l)
    if result["disease"] is None: result["disease"]=_match_disease(clean_l)
    if result["diseased"] is None and result["disease"]:
        result["diseased"]=0 if result["disease"]==HEALTHY_LABEL else 1
    if result["confidence"] is None:
        m=re.search(r"confidence[\s:]*([0-9]*\.?[0-9]+)",clean_l)
        if m:
            c=float(m.group(1)); result["confidence"]=c/100 if c>1 else c
    if result["crop"] is None and agent_name:
        print(f"   [WARN] [{agent_name}] CROP parse FAILED: {repr(str(text)[:80])}")
    if result["disease"] is None and agent_name:
        print(f"   [WARN] [{agent_name}] DISEASE parse FAILED: {repr(str(text)[:80])}")
    return result

print("Parser    : defined (3-pass matching)")

## Cell 4: Agent Inference Functions

In [ ]:
def encode_b64(path):
    with open(path,"rb") as f: return base64.b64encode(f.read()).decode()

def mime_type(path):
    return "image/jpeg" if str(path).lower().endswith(("jpg","jpeg")) else "image/png"

# Dataset name withheld unless MENTION_DATASET_NAME=True (anti-leakage, reviewer R2-Meth1)
_DATASET_TAG = "from the CDDM crop disease dataset" if MENTION_DATASET_NAME else "from crop leaf photographs"
_SYSTEM_PROMPT = (
    f"You are an expert plant pathologist classifying crop leaf images {_DATASET_TAG}. "
    f"Valid crops (choose EXACTLY one): {_CROP_LIST}. "
    f"Valid diseases (choose EXACTLY one): {_DISEASE_LIST}. "
    "NEVER output a crop or disease not in these lists. "
    "If unsure, pick the closest match. "
    "Do NOT ask for clarification — always classify."
)

def _openai_call(client, model, image_path, prompt, retries=3, delay=5):
    b64=encode_b64(image_path)
    for attempt in range(retries):
        try:
            resp=client.chat.completions.create(
                model=model,
                messages=[
                    {"role":"system","content":_SYSTEM_PROMPT},
                    {"role":"user","content":[
                        {"type":"image_url","image_url":{
                            "url":f"data:{mime_type(image_path)};base64,{b64}",
                            "detail":"high"}},
                        {"type":"text","text":prompt}
                    ]}
                ],
                max_tokens=600,
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            if attempt<retries-1:
                print(f"   [WARN] [{model}] ({str(e)[:80]}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

def call_openai(model, image_path, prompt, retries=3, delay=5):
    return _openai_call(openai_client, model, image_path, prompt, retries, delay)

def call_xai(model, image_path, prompt, retries=3, delay=5):
    return _openai_call(xai_client, model, image_path, prompt, retries, delay)

def call_anthropic(model, image_path, prompt, retries=3, delay=5):
    b64=encode_b64(image_path); mt=mime_type(image_path)
    for attempt in range(retries):
        try:
            resp=anthropic_client.messages.create(
                model=model, max_tokens=600,
                system=_SYSTEM_PROMPT,
                messages=[{"role":"user","content":[
                    {"type":"image","source":{"type":"base64","media_type":mt,"data":b64}},
                    {"type":"text","text":prompt}
                ]}])
            return resp.content[0].text.strip()
        except Exception as e:
            if attempt<retries-1:
                print(f"   [WARN] [Anthropic/{model}] ({str(e)[:80]}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

_url_cache = {}

def _upload_to_url(image_path, retries=2):
    img_bytes=open(image_path,"rb").read()
    img_name=Path(image_path).name; img_mime=mime_type(image_path)
    hosts = [
        ("catbox",    lambda: _requests.post("https://catbox.moe/user/api.php",
            data={"reqtype":"fileupload"},
            files={"fileToUpload":(img_name,img_bytes,img_mime)},timeout=30)),
        ("litterbox", lambda: _requests.post(
            "https://litterbox.catbox.moe/resources/internals/api.php",
            data={"reqtype":"fileupload","time":"72h"},
            files={"fileToUpload":(img_name,img_bytes,img_mime)},timeout=30)),
        ("tmpfiles",  lambda: _requests.post("https://tmpfiles.org/api/v1/upload",
            files={"file":(img_name,img_bytes,img_mime)},timeout=30)),
    ]
    for host_name, fn in hosts:
        for attempt in range(retries):
            try:
                r=fn()
                if r.status_code==200:
                    url=(r.json().get("data",{}).get("url","").replace(
                        "tmpfiles.org/","tmpfiles.org/dl/")
                        if host_name=="tmpfiles" else r.text.strip())
                    if url.startswith("https://"): return url
            except Exception: pass
            time.sleep(2)
    print(f"   [ERROR] All image hosts failed for {Path(image_path).name}")
    return None

def call_together(model, image_path, prompt, retries=3, delay=5):
    img_key=str(image_path)
    if img_key not in _url_cache:
        url=_upload_to_url(image_path)
        if url is None: return "ERROR: image upload failed"
        _url_cache[img_key]=url
    img_url=_url_cache[img_key]
    for attempt in range(retries):
        try:
            resp=_together_compat.chat.completions.create(
                model=model,
                messages=[
                    {"role":"system","content":_SYSTEM_PROMPT},
                    {"role":"user","content":[
                        {"type":"image_url","image_url":{"url":img_url}},
                        {"type":"text","text":prompt},
                    ]}
                ],
                max_tokens=700,timeout=90.0,
                # Qwen3.5/3.6 think by default on Together -> disable, else empty content
                **({"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
                   if ("qwen3.5" in model.lower() or "qwen3.6" in model.lower()) else {}),
            )
            raw=(resp.choices[0].message.content or "").strip()
            if not raw:
                if attempt<retries-1:
                    _url_cache.pop(img_key,None)
                    new_url=_upload_to_url(image_path)
                    if new_url: _url_cache[img_key]=img_url=new_url
                    print(f"   [WARN] [{model}] empty, retry {attempt+1} (fresh URL)...")
                    time.sleep(delay*2); continue
                else: return f"ERROR: {model} empty after {retries} retries"
            return raw
        except Exception as e:
            err=str(e)
            if "corrupt" in err.lower() or "could not be processed" in err.lower():
                _url_cache.pop(img_key,None)
                new_url=_upload_to_url(image_path)
                if new_url: _url_cache[img_key]=img_url=new_url
            if attempt<retries-1:
                print(f"   [WARN] [Together/{model}] ({err[:80]}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {err}"

AGENT_FN_MAP = {
    "GPT-4.1":        lambda img,p: call_openai("gpt-4.1",img,p),
    "GPT-4.1-mini":   lambda img,p: call_openai("gpt-4.1-mini",img,p),
    "Claude-Sonnet":  lambda img,p: call_anthropic("claude-sonnet-4-5",img,p),
    "Claude-Haiku":   lambda img,p: call_anthropic("claude-haiku-4-5",img,p),
    "Grok-4.20-Fast": lambda img,p: call_xai("grok-4.20-non-reasoning",img,p),
    "Gemma-3n-E4B":   lambda img,p: call_together("google/gemma-3n-E4B-it",img,p),
    "Qwen-VL":        lambda img,p: call_together(QWEN_MODEL,img,p),   # open-source, debate-eligible
}

print(f"Agents    : {list(AGENT_FN_MAP.keys())}")

## Cell 5: Debate Prompts & Softmax Voting

In [ ]:
_FMT_R2 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <address peers by name, cite specific visual features>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict, or None>\n"
    "New Visual Evidence: <specific new visual feature you now observe, or NONE>"
)
_FMT_R3 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <final position, address peers by name>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict vs R2, or None>\n"
    "New Visual Evidence: <specific new visual feature observed vs R2, or NONE>"
)

def make_r2_prompt(my_name, own_r1, others_r1):
    own=(f"YOUR Round 1 verdict:\n"
         f"  Crop: {own_r1.get('crop','?')} / Disease: {own_r1.get('disease','?')}\n"
         f"  Reasoning: {own_r1.get('reasoning','')[:120]}")
    peers="".join(
        f"\n[{pn}]: {p.get('crop','?')} / {p.get('disease','?')}\n"
        f"  Reasoning: {p.get('reasoning','')[:120]}\n"
        for pn,p in others_r1.items())
    return (f"You are {my_name}, a plant science researcher.\n\n{own}\n\n"
            f"Peers analyzed the SAME image:\n{peers}\n"
            f"Re-examine the image. Address {' and '.join(others_r1.keys())} by name. "
            f"AGREE or DISAGREE citing visual features. "
            f"Change verdict ONLY if you observe new visual evidence.\n\n"+_FMT_R2)

def make_r3_prompt(my_name, own_r1, own_r2, others_r2):
    history=(f"YOUR history: R1={own_r1.get('crop','?')}/{own_r1.get('disease','?')} "
             f"-> R2={own_r2.get('crop','?')}/{own_r2.get('disease','?')}\n"
             f"  R2 Reasoning: {own_r2.get('reasoning','')[:120]}")
    peers="".join(
        f"\n[{pn}] R2: {p.get('crop','?')}/{p.get('disease','?')}\n"
        f"  Reasoning: {p.get('reasoning','')[:120]}\n"
        for pn,p in others_r2.items())
    return (f"You are {my_name}. FINAL ROUND.\n\n{history}\n\nPeers R2:\n{peers}\n"
            f"Commit to your FINAL verdict. Address {' and '.join(others_r2.keys())} by name.\n\n"
            +_FMT_R3)

def compute_agent_weights(agent_names, metrics_df):
    accs=np.array([float(metrics_df.loc[a,"Combined Acc"])
                   if a in metrics_df.index else 0.0 for a in agent_names])
    exp_a=np.exp(accs/SOFTMAX_TEMP); weights=exp_a/exp_a.sum()
    w_dict=dict(zip(agent_names,weights))
    print("   Voting weights (softmax T=2.0):")
    for n,w in sorted(w_dict.items(),key=lambda x:-x[1]):
        acc=float(metrics_df.loc[n,"Combined Acc"]) if n in metrics_df.index else 0.0
        print(f"      {n:<22}: acc={acc:.3f} -> weight={w:.3f}")
    return w_dict

def weighted_vote(values_dict, weights_dict):
    scores={}
    for agent,val in values_dict.items():
        if val is None: continue
        scores[val]=scores.get(val,0.0)+weights_dict.get(agent,1.0/max(len(values_dict),1))
    if not scores: return None,{},0.0
    winner=max(scores,key=scores.get)
    return winner,scores,scores[winner]

def simple_vote(values):
    valid=[v for v in values if v is not None]
    if not valid: return None,{},0.0
    counts=Counter(valid); top=counts.most_common()
    if len(top)==1 or top[0][1]>top[1][1]:
        w=top[0][0]; return w,dict(counts),counts[w]/len(valid)
    return sorted(top,key=lambda x:(-x[1],str(x[0])))[0][0],dict(counts),top[0][1]/len(valid)

def aggregate_votes(parsed_dict, weights_dict, round_label="r3"):
    agents=list(parsed_dict.keys())
    crop_w,_,crop_aw=weighted_vote({a:parsed_dict[a]["crop"]    for a in agents},weights_dict)
    dis_w, _,dis_aw =weighted_vote({a:parsed_dict[a]["disease"] for a in agents},weights_dict)
    dis_b, _,_      =weighted_vote({a:parsed_dict[a]["diseased"] for a in agents},weights_dict)
    confs=[parsed_dict[a]["confidence"] for a in agents if parsed_dict[a]["confidence"] is not None]
    maj=[a for a in agents if parsed_dict[a]["disease"]==dis_w]
    reasoning=" | ".join(parsed_dict[a]["reasoning"] for a in maj if parsed_dict[a]["reasoning"])
    pfx=f"{round_label}_"
    return {
        f"{pfx}consensus_crop":    crop_w,
        f"{pfx}consensus_disease": dis_w,
        f"{pfx}consensus_diseased":dis_b,
        f"{pfx}consensus_reasoning":reasoning,
        f"{pfx}mean_confidence":   float(np.mean(confs)) if confs else None,
        f"{pfx}crop_vote_agree_w": round(crop_aw,3),
        f"{pfx}dis_vote_agree_w":  round(dis_aw,3),
        f"{pfx}individual_crops":    {a:parsed_dict[a]["crop"]    for a in agents},
        f"{pfx}individual_diseases": {a:parsed_dict[a]["disease"] for a in agents},
    }

def aggregate_debate(r1_p,r2_p,r3_p,weights_dict=None):
    if weights_dict is None: weights_dict={a:1.0/len(r3_p) for a in r3_p}
    result={}
    result.update(aggregate_votes(r3_p,weights_dict,round_label="r3"))
    result.update(aggregate_votes(r1_p,weights_dict,round_label="r1_baseline"))
    result["consensus_crop"]         =result["r3_consensus_crop"]
    result["consensus_disease"]      =result["r3_consensus_disease"]
    result["consensus_diseased"]     =result["r3_consensus_diseased"]
    result["consensus_reasoning"]    =result["r3_consensus_reasoning"]
    result["mean_confidence"]        =result["r3_mean_confidence"]
    result["crop_vote_agreement"]    =result["r3_crop_vote_agree_w"]
    result["disease_vote_agreement"] =result["r3_dis_vote_agree_w"]
    result["individual_crops_r3"]    =result["r3_individual_crops"]
    result["individual_diseases_r3"] =result["r3_individual_diseases"]
    return result

print("Prompts   : defined (R2 named debate, R3 final verdict)")
print("Voting    : defined (softmax-weighted + pre-debate baseline)")

## Cell 6: File Debug — Verify VIDA Inputs
Run this before anything else. It searches all uploaded paths, loads the files,
reconstructs the pilot list, and spot-checks parsing. Fix any red lines before proceeding.

In [ ]:
# ── Quick path listing — shows all non-image files in /kaggle/input ──────────
print("Scanning /kaggle/input for CSV and JSON files...")
from pathlib import Path
import re

_found_files = []
def _walk(path, depth=0, max_depth=6):
    if depth > max_depth: return
    try:
        for p in sorted(path.iterdir()):
            if p.is_file() and p.suffix.lower() in (".csv",".json"):
                print(f"  {'  '*depth}{p}")
                _found_files.append(p)
            elif p.is_dir() and p.name not in ("images","color","segmented","grayscale"):
                _walk(p, depth+1, max_depth)
    except PermissionError:
        pass

_walk(Path("/kaggle/input"))
if not _found_files:
    print("  No CSV/JSON files found yet — dataset may still be downloading")
print()

# ══════════════════════════════════════════════════════════════════════════════
#  FILE DEBUG CELL — verify all VIDA output files are readable
# ══════════════════════════════════════════════════════════════════════════════

SEP  = "=" * 65
THIN = "-" * 65

def _find_file(filename):
    """Find file by stem, skipping image folders to keep it fast."""
    stem    = Path(filename).stem
    suffix  = Path(filename).suffix
    pattern = re.compile(r"^" + re.escape(stem) + r"(\s*\(\d+\))?" + re.escape(suffix) + r"$", re.IGNORECASE)
    skip_dirs = {"images","color","segmented","grayscale"}

    def _search(base, depth=0):
        if depth > 6: return None
        try:
            for p in base.iterdir():
                if p.is_file() and pattern.match(p.name): return p
                if p.is_dir() and p.name not in skip_dirs:
                    result = _search(p, depth+1)
                    if result: return result
        except PermissionError:
            pass
        return None

    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        result = _search(base)
        if result: return result
    return None

_rp = _find_file("r1_results.csv")
_mp = _find_file("r1_metrics.csv")   # OPTIONAL now — metrics are rebuilt from r1_df below
_bp = _find_file("r1_raw_bank.json")

_all_found = True
for fname, fpath, required in [("r1_results.csv",  _rp, True),
                               ("r1_raw_bank.json", _bp, True),
                               ("r1_metrics.csv",   _mp, False)]:
    if fpath:
        size_kb = fpath.stat().st_size / 1024
        print(f"  [OK] {fname}")
        print(f"       Path : {fpath}")
        print(f"       Size : {size_kb:.1f} KB")
    elif required:
        print(f"  [!!] {fname} — NOT FOUND (required)")
        print(f"       Upload via Kaggle sidebar -> Add Input -> Upload Dataset")
        _all_found = False
    else:
        print(f"  [--] {fname} — not found (optional; metrics rebuilt from r1_results.csv)")

if not _all_found:
    print()
    print("  STOP: Upload the missing REQUIRED file(s) and re-run this cell.")
    raise FileNotFoundError("Missing required VIDA outputs (r1_results.csv / r1_raw_bank.json).")

# ── Load files ────────────────────────────────────────────────────────────────
print()
print(SEP)
print("  LOADING & VALIDATING")
print(SEP)

r1_df      = pd.read_csv(_rp)
with open(_bp) as f: r1_raw_bank = json.load(f)

# ── Rebuild metrics_df from r1_df (source of truth) ──────────────────────────
# Guarantees ALL agents are present (incl. Qwen-VL) and never stale. A loaded
# r1_metrics.csv (if any) only contributes EXTRA columns (RQ / Plausibility).
metrics_df = r1_df.groupby("agent").agg(**{
    "Crop Acc":     ("crop_correct",    "mean"),
    "Disease Acc":  ("disease_correct", "mean"),
    "Combined Acc": ("both_correct",    "mean"),
    "Parse Fail %": ("parse_failed",    lambda x: 100.0 * x.mean()),
})
metrics_df.index.name = "Agent"
if _mp is not None:
    try:
        _old = pd.read_csv(_mp, index_col="Agent")
        for col in _old.columns:                       # keep r1_df accuracies authoritative
            if col not in metrics_df.columns:
                metrics_df[col] = _old[col]
    except Exception as e:
        print(f"  (note: could not read optional r1_metrics.csv: {e})")
metrics_df = metrics_df.sort_values("Combined Acc", ascending=False)

# Copy source files to output dir; write a fresh, complete metrics file
import shutil
for src_p, fname in [(_rp,"r1_results.csv"),(_bp,"r1_raw_bank.json")]:
    dest = OUTPUT_DIR / fname
    if str(src_p) != str(dest): shutil.copy2(src_p, dest)
metrics_df.to_csv(OUTPUT_DIR / "r1_metrics.csv")

vida_agent_names = sorted(r1_df["agent"].unique().tolist())
unique_imgs      = r1_df["image_name"].unique().tolist()

print(f"  r1_df        : {r1_df.shape[0]:,} rows x {r1_df.shape[1]} cols")
print(f"  metrics_df   : {metrics_df.shape[0]} agents")
print(f"  r1_raw_bank  : {len(r1_raw_bank):,} images")
print(f"  Agents found : {vida_agent_names}")
print(f"  Images       : {len(unique_imgs):,}")
print()

# ── Metrics summary ───────────────────────────────────────────────────────────
print(SEP)
print("  VIDA METRICS SUMMARY (rebuilt from r1_results.csv — all agents)")
print(SEP)
_show_cols = [c for c in ["Crop Acc","Disease Acc","Combined Acc","Parse Fail %",
                           "Reasoning Quality","Plausibility"] if c in metrics_df.columns]
_sorted = metrics_df[_show_cols].sort_values("Combined Acc", ascending=False)
for agent, row in _sorted.iterrows():
    vals = "  ".join(f"{col.split()[0][:4]}={row[col]:.3f}" if pd.notna(row[col]) else f"{col[:4]}=N/A"
                     for col in _show_cols)
    print(f"  {agent:<22}: {vals}")

# ── Reconstruct pilot list ────────────────────────────────────────────────────
print()
print(SEP)
print("  RECONSTRUCTING PILOT (image paths from CDDM dataset)")
print(SEP)

_pilot_meta = r1_df.drop_duplicates("image_name")[
    ["image_name","gt_crop","gt_disease","gt_diseased","class_label"]].copy()

pilot = []
_missing_imgs = []
for _, row in _pilot_meta.iterrows():
    folder    = row["class_label"]
    img_path  = DATASET_ROOT / folder / row["image_name"] if DATASET_ROOT else None
    if img_path and not img_path.exists():
        matches = list(DATASET_ROOT.rglob(row["image_name"])) if DATASET_ROOT else []
        img_path = matches[0] if matches else img_path
    if img_path is None or not img_path.exists():
        _missing_imgs.append(row["image_name"])
    pilot.append({
        "image_path":  img_path,
        "image_name":  row["image_name"],
        "gt_crop":     row["gt_crop"],
        "gt_disease":  row["gt_disease"],
        "gt_diseased": int(row["gt_diseased"]),
        "class_label": row["class_label"],
    })

print(f"  Total pilot  : {len(pilot)} images")
print(f"  Found on disk: {len(pilot) - len(_missing_imgs)}")
if _missing_imgs:
    print(f"  Missing      : {len(_missing_imgs)} (debate will skip these)")
    for m in _missing_imgs[:5]: print(f"    - {m}")
    if len(_missing_imgs) > 5: print(f"    ... and {len(_missing_imgs)-5} more")
else:
    print(f"  Missing      : 0 — all images found")

# ── Raw bank spot-check ───────────────────────────────────────────────────────
print()
print(SEP)
print("  R1 RAW BANK SPOT-CHECK (3 random images, first VIDA agent)")
print(SEP)

_spot = random.sample(unique_imgs, min(3, len(unique_imgs)))
_first_agent = vida_agent_names[0]
for img_name in _spot:
    raw  = r1_raw_bank.get(img_name, {}).get(_first_agent, "")
    p    = parse_response(raw)
    ok   = "[OK]" if (p["crop"] and p["disease"]) else "[PARSE FAIL]"
    print(f"  {ok} {img_name[:45]}")
    print(f"       Agent={_first_agent} | Crop={p['crop']} | Disease={p['disease']}")

print()
print(SEP)
if _missing_imgs:
    print(f"  [OK] Files loaded — {len(_missing_imgs)} images missing from CDDM dataset")
else:
    print(f"  [OK] All files loaded successfully — proceed to Cell 7")
print(SEP)

## Cell 7: Agent Selection for PANDA Debate

Review the metrics summary from Cell 6.
Set `AUTO_SELECT=True` for automatic top-N by Combined Accuracy,
or `AUTO_SELECT=False` and fill `MANUAL_AGENTS`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
AUTO_SELECT   = False    # True = auto top-N by Combined Acc | False = manual
N_AUTO        = 3       # number of debate agents (3 recommended)

MANUAL_AGENTS = [       # used when AUTO_SELECT = False  (Qwen-VL / Gemma-3n-E4B selectable for open setups)
    
    "GPT-4.1",
    "Claude-Sonnet",
    
    "Grok-4.20-Fast",
    
]
# ══════════════════════════════════════════════════════════════════════════════

if AUTO_SELECT:
    DEBATE_AGENTS = metrics_df.sort_values("Combined Acc",ascending=False).head(N_AUTO).index.tolist()
    # Filter to only agents that exist in AGENT_FN_MAP (can debate)
    DEBATE_AGENTS = [a for a in DEBATE_AGENTS if a in AGENT_FN_MAP]
    print(f"[OK] AUTO-SELECTED top-{N_AUTO}: {DEBATE_AGENTS}")
else:
    invalid = [a for a in MANUAL_AGENTS if a not in AGENT_FN_MAP]
    if invalid: raise ValueError(f"Invalid agents: {invalid}. Valid: {list(AGENT_FN_MAP.keys())}")
    DEBATE_AGENTS = MANUAL_AGENTS
    print(f"[OK] MANUAL selection: {DEBATE_AGENTS}")

# Check all selected agents have R1 data
_missing_r1 = [a for a in DEBATE_AGENTS if a not in vida_agent_names]
if _missing_r1:
    print(f"[WARN] These agents have no R1 data: {_missing_r1}")
    print(f"       Available in r1_raw_bank: {vida_agent_names}")

print(f"\n  Selected agents R1 stats:")
_show = [c for c in ["Crop Acc","Disease Acc","Combined Acc"] if c in metrics_df.columns]
print(metrics_df.loc[[a for a in DEBATE_AGENTS if a in metrics_df.index], _show].to_string())

AGENT_WEIGHTS = compute_agent_weights(DEBATE_AGENTS, metrics_df)
AGENTS_DEBATE = {name: AGENT_FN_MAP[name] for name in DEBATE_AGENTS if name in AGENT_FN_MAP}
debate_agents_list = list(AGENTS_DEBATE.keys())
print(f"\n[OK] PANDA ready — {len(debate_agents_list)} agents, weights computed")

## Cell 7b: Qwen VLM Debug & Fallback  (only if Qwen-VL is a debate agent)

In [ ]:
# If Qwen-VL is among the selected debate agents, validate the primary QWEN_MODEL and
# fall back through QWEN_FALLBACKS before the (live) R2/R3 debate calls. Updates the
# global QWEN_MODEL; the Qwen-VL agent lambda picks it up via late binding.
if "Qwen-VL" in DEBATE_AGENTS:
    _img = next((s["image_path"] for s in pilot
                 if s["image_path"] is not None and s["image_path"].exists()), None)
    assert _img is not None, "No image found on disk to test Qwen — check DATASET_ROOT"
    _qprompt = (
        "You are an expert plant pathologist. Analyze this leaf image.\n\n"
        "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
        f"Crop Category: <one of: {_CROP_LIST}>\n"
        f"Disease Category: <one of: {_DISEASE_LIST}>\n"
        "Diseased: <Yes or No>\n"
        "Reasoning: <one sentence>\n"
        "Confidence: <0.0 to 1.0>"
    )
    print("=" * 65); print("  QWEN VLM DEBUG & FALLBACK"); print("=" * 65)
    _candidates = [QWEN_MODEL] + [m for m in QWEN_FALLBACKS if m != QWEN_MODEL]
    _qwen_ok = False
    for _cand in _candidates:
        print(f"\n▶ Testing: {_cand}")
        try:
            _raw = call_together(_cand, _img, _qprompt)
            if (not _raw) or _raw.startswith("ERROR") or not _raw.strip():
                print(f"   [X] failed/empty: {repr((_raw or '')[:120])}"); time.sleep(2); continue
            _p = parse_response(_raw, agent_name=f"TEST-{_cand}")
            if _p["crop"] or _p["disease"]:
                QWEN_MODEL = _cand
                print(f"   [OK] WORKING — using {QWEN_MODEL}")
                print(f"        Crop={_p['crop']} | Disease={_p['disease']} | Conf={_p['confidence']}")
                _qwen_ok = True; break
            else:
                print(f"   [!] parse fail: {repr(_raw[:200])}")
        except Exception as e:
            print(f"   [X] exception: {e}")
        time.sleep(2)
    print("\n" + "=" * 65)
    if _qwen_ok:
        print(f"  [OK] Qwen-VL resolved to: {QWEN_MODEL}")
    else:
        print("  [X] No working Qwen VLM. Add a candidate to QWEN_FALLBACKS, or drop Qwen-VL")
        print("      from MANUAL_AGENTS in the previous cell and re-run.")
    print("=" * 65)
else:
    print("Qwen-VL not in DEBATE_AGENTS — skipping Qwen resolver.")


## Cell 8: Pre-PANDA Debate Test
Runs one full R2+R3 debate on a single image. Send output before running Cell 9.

In [ ]:
# Find a test image where debate agents disagreed in R1
_test_sample = None
for sample in pilot[:60]:
    img_name = sample["image_name"]
    if img_name not in r1_raw_bank: continue
    if sample["image_path"] is None or not sample["image_path"].exists(): continue
    r1_test = {n: parse_response(r1_raw_bank[img_name].get(n,"")) for n in DEBATE_AGENTS}
    unique = len(set(r1_test[n]["crop"] for n in DEBATE_AGENTS if r1_test[n]["crop"]))
    if unique >= 2: _test_sample = sample; break
if _test_sample is None:
    _test_sample = next((s for s in pilot
                         if s["image_name"] in r1_raw_bank
                         and s["image_path"] and s["image_path"].exists()), None)

assert _test_sample is not None, "No usable test image found in pilot"

img_path   = _test_sample["image_path"]
img_name   = _test_sample["image_name"]
gt_crop    = _test_sample["gt_crop"]
gt_disease = _test_sample["gt_disease"]

print("=" * 70)
print("  PRE-PANDA DEBATE TEST")
print("=" * 70)
print(f"  Image   : {img_name}")
print(f"  GT      : {gt_crop} / {gt_disease}")
print(f"  Agents  : {DEBATE_AGENTS}\n")

r1_p = {n: parse_response(r1_raw_bank[img_name].get(n,"")) for n in DEBATE_AGENTS}
print("  R1 (pre-debate):")
for n in DEBATE_AGENTS:
    p = r1_p[n]
    ok = "[OK]" if (p["crop"]==gt_crop and p["disease"]==gt_disease)          else ("[CROP]" if p["crop"]==gt_crop else "[X]")
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} conf={p['confidence']}")

print("\n  R2 (named debate):")
r2_raw={}; r2_p={}
for n,fn in AGENTS_DEBATE.items():
    others={k:v for k,v in r1_p.items() if k!=n}
    raw=fn(img_path,make_r2_prompt(n,r1_p[n],others))
    r2_raw[n]=raw; r2_p[n]=parse_response(raw,agent_name=f"TEST-R2-{n}")
    p=r2_p[n]
    ok="[OK]" if (p["crop"]==gt_crop and p["disease"]==gt_disease)        else ("[CROP]" if p["crop"]==gt_crop else "[X]")
    inf  = p["influenced_by"] or "None"
    ev   = (p["new_visual_evidence"] or "NONE")[:40]
    chng = " <-- CHANGED" if (r1_p[n]["disease"]!=p["disease"] and
                               r1_p[n]["disease"] and p["disease"]) else ""
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} | inf={inf} | ev={ev}{chng}")
    print(f"       {(p['reasoning'] or '')[:100]}")

print("\n  R3 (final verdict):")
r3_raw={}; r3_p={}
for n,fn in AGENTS_DEBATE.items():
    others={k:v for k,v in r2_p.items() if k!=n}
    raw=fn(img_path,make_r3_prompt(n,r1_p[n],r2_p[n],others))
    r3_raw[n]=raw; r3_p[n]=parse_response(raw,agent_name=f"TEST-R3-{n}")
    p=r3_p[n]
    ok="[OK]" if (p["crop"]==gt_crop and p["disease"]==gt_disease)        else ("[CROP]" if p["crop"]==gt_crop else "[X]")
    print(f"    {ok} {n:<22}: {p['crop']}/{p['disease']} conf={p['confidence']} inf={p['influenced_by']}")

cons=aggregate_debate(r1_p,r2_p,r3_p,weights_dict=AGENT_WEIGHTS)
b_ok=int(cons.get("r1_baseline_consensus_crop","")==gt_crop and
         cons.get("r1_baseline_consensus_disease","")==gt_disease)
p_ok=int(cons["consensus_crop"]==gt_crop and cons["consensus_disease"]==gt_disease)
print(f"\n  PRE-DEBATE  consensus: {cons.get('r1_baseline_consensus_crop','?')}/{cons.get('r1_baseline_consensus_disease','?')} {'[OK]' if b_ok else '[X]'}")
print(f"  POST-DEBATE consensus: {cons['consensus_crop']}/{cons['consensus_disease']} {'[OK]' if p_ok else '[X]'}")
print("\n  Send this output to verify debate format, then run Cell 9")
print("=" * 70)

## Cell 9: PANDA Block — Run R2 + R3 Debate

In [ ]:
import os, glob, re

# ── Resume from checkpoint (works across Kaggle sessions) ─────────────────────
# /kaggle/working is wiped between sessions. If you uploaded the checkpoint files
# as a Kaggle dataset, this finds them under /kaggle/input and resumes. New
# checkpoints keep saving to OUTPUT_DIR — re-upload the latest to resume again.
_debate_ckpt = OUTPUT_DIR / "debate_results_checkpoint.csv"
_trans_ckpt  = OUTPUT_DIR / "debate_transcripts_checkpoint.json"

def _locate_ckpt(working_path, stem):
    """Prefer the in-session working copy; else shallow-search /kaggle/input.
    Uses fixed-depth globs (NOT recursive) to avoid scanning the huge image dataset."""
    if working_path.exists():
        return working_path
    for d in range(1, 5):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * d + stem + "*"))
        if hits:
            return Path(hits[0])
    return None

_csv_src = _locate_ckpt(_debate_ckpt, "debate_results_checkpoint")
_jsn_src = _locate_ckpt(_trans_ckpt,  "debate_transcripts_checkpoint")

if _csv_src is not None:
    _ex            = pd.read_csv(_csv_src)
    _done_debate   = set(_ex["image_name"].tolist())
    debate_results = _ex.to_dict("records")
    print(f"Resuming from checkpoint: {len(_done_debate)} images already done")
    print(f"   results     : {_csv_src}")
    # Safety: selected debate agents MUST match those baked into the checkpoint
    _ckpt_agents = sorted({m.group(1) for c in _ex.columns
                           for m in [re.match(r'r1_(.+?)_crop$', c)]
                           if m and m.group(1) != "baseline_consensus"})
    if _ckpt_agents != sorted(debate_agents_list):
        raise ValueError(
            f"Agent mismatch: checkpoint ran with {_ckpt_agents}, but current selection "
            f"is {sorted(debate_agents_list)}. Set the SAME lineup in the agent-selection "
            f"cell before resuming (or remove the checkpoint to start fresh).")
else:
    _done_debate = set(); debate_results = []
    print("No checkpoint found — starting fresh")

if _jsn_src is not None:
    debate_transcripts = json.load(open(_jsn_src))
    print(f"   transcripts : {_jsn_src}")
else:
    debate_transcripts = {}

_remaining = [s for s in pilot if s["image_name"] not in _done_debate
              and s["image_path"] is not None and s["image_path"].exists()]

# ── Atomic checkpoint saver ───────────────────────────────────────────────────
# Writes to a .tmp file then os.replace() (atomic on the same filesystem), so a
# kernel death mid-save can never leave a half-written / corrupt checkpoint.
def _save_ckpt(tag=""):
    _tmp_csv = _debate_ckpt.parent / (_debate_ckpt.name + ".tmp")
    _tmp_jsn = _trans_ckpt.parent  / (_trans_ckpt.name  + ".tmp")
    pd.DataFrame(debate_results).to_csv(_tmp_csv, index=False)
    os.replace(_tmp_csv, _debate_ckpt)
    with open(_tmp_jsn, "w") as f: json.dump(debate_transcripts, f)
    os.replace(_tmp_jsn, _trans_ckpt)
    if tag: print(f"   [checkpoint: {tag}] {len(debate_results)} images saved")

print(f"PANDA: {len(_remaining)} remaining x {len(debate_agents_list)} agents x 2 rounds")
print(f"Agents: {debate_agents_list}")
print(f"Checkpoint every {CHECKPOINT_EVERY} images (+ on interrupt / error / finish)\n")

_processed  = 0
_completed  = False
try:
    for i, sample in enumerate(tqdm(_remaining, desc="PANDA")):
        img_path    = sample["image_path"]
        img_name    = sample["image_name"]
        gt_crop     = sample["gt_crop"]
        gt_disease  = sample["gt_disease"]
        gt_diseased = sample["gt_diseased"]

        print(f"\n[{len(_done_debate)+i+1}/{len(pilot)}] {img_name} | GT: {gt_crop}/{gt_disease}")

        r1_p = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,""))
                for n in debate_agents_list}
        result = {"image_name":img_name,"class_label":sample["class_label"],
                  "gt_crop":gt_crop,"gt_disease":gt_disease,"gt_diseased":gt_diseased}
        for n in debate_agents_list:
            result[f"r1_{n}_crop"]=r1_p[n]["crop"]
            result[f"r1_{n}_disease"]=r1_p[n]["disease"]

        # R2
        print("  [R2] Named debate...")
        r2_raw={}; r2_p={}
        for n,fn in AGENTS_DEBATE.items():
            others={k:v for k,v in r1_p.items() if k!=n}
            raw=fn(img_path,make_r2_prompt(n,r1_p[n],others))
            r2_raw[n]=raw; r2_p[n]=parse_response(raw,agent_name=f"{n} R2")
            result[f"r2_{n}_crop"]            = r2_p[n]["crop"]
            result[f"r2_{n}_disease"]         = r2_p[n]["disease"]
            result[f"r2_{n}_influenced_by"]   = r2_p[n]["influenced_by"]
            result[f"r2_{n}_new_evidence"]    = r2_p[n]["new_visual_evidence"]
            result[f"r2_{n}_crop_changed"]    = int(bool(r1_p[n]["crop"]) and bool(r2_p[n]["crop"]) and r1_p[n]["crop"]!=r2_p[n]["crop"])
            result[f"r2_{n}_disease_changed"] = int(bool(r1_p[n]["disease"]) and bool(r2_p[n]["disease"]) and r1_p[n]["disease"]!=r2_p[n]["disease"])
            print(f"    {n}: {r2_p[n]['crop']}/{r2_p[n]['disease']} | inf={r2_p[n]['influenced_by']}")

        # R3
        print("  [R3] Final verdict...")
        r3_raw={}; r3_p={}
        for n,fn in AGENTS_DEBATE.items():
            others={k:v for k,v in r2_p.items() if k!=n}
            raw=fn(img_path,make_r3_prompt(n,r1_p[n],r2_p[n],others))
            r3_raw[n]=raw; r3_p[n]=parse_response(raw,agent_name=f"{n} R3")
            result[f"r3_{n}_crop"]            = r3_p[n]["crop"]
            result[f"r3_{n}_disease"]         = r3_p[n]["disease"]
            result[f"r3_{n}_confidence"]      = r3_p[n]["confidence"]
            result[f"r3_{n}_influenced_by"]   = r3_p[n]["influenced_by"]
            result[f"r3_{n}_new_evidence"]    = r3_p[n]["new_visual_evidence"]
            result[f"r3_{n}_crop_changed"]    = int(bool(r2_p[n]["crop"]) and bool(r3_p[n]["crop"]) and r2_p[n]["crop"]!=r3_p[n]["crop"])
            result[f"r3_{n}_disease_changed"] = int(bool(r2_p[n]["disease"]) and bool(r3_p[n]["disease"]) and r2_p[n]["disease"]!=r3_p[n]["disease"])
            print(f"    {n}: {r3_p[n]['crop']}/{r3_p[n]['disease']} conf={r3_p[n]['confidence']} inf={r3_p[n]['influenced_by']}")

        cons=aggregate_debate(r1_p,r2_p,r3_p,weights_dict=AGENT_WEIGHTS)
        result.update(cons)
        result["consensus_crop_correct"]    = int(cons["consensus_crop"]==gt_crop)    if cons["consensus_crop"]    else 0
        result["consensus_disease_correct"] = int(cons["consensus_disease"]==gt_disease) if cons["consensus_disease"] else 0
        result["consensus_both_correct"]    = int(result["consensus_crop_correct"] and result["consensus_disease_correct"])
        result["baseline_crop_correct"]     = int(cons.get("r1_baseline_consensus_crop","")==gt_crop)
        result["baseline_disease_correct"]  = int(cons.get("r1_baseline_consensus_disease","")==gt_disease)
        result["baseline_both_correct"]     = int(result["baseline_crop_correct"] and result["baseline_disease_correct"])
        for n in debate_agents_list:
            for rnd,p in [("r1",r1_p),("r2",r2_p),("r3",r3_p)]:
                result[f"{rnd}_{n}_crop_correct"]    = int(p[n]["crop"]==gt_crop)     if p[n]["crop"]    else 0
                result[f"{rnd}_{n}_disease_correct"] = int(p[n]["disease"]==gt_disease) if p[n]["disease"] else 0
                result[f"{rnd}_{n}_both_correct"]    = int(result[f"{rnd}_{n}_crop_correct"] and result[f"{rnd}_{n}_disease_correct"])

        debate_results.append(result)
        debate_transcripts[img_name]={"r2":r2_raw,"r3":r3_raw}
        _processed += 1

        if _processed % CHECKPOINT_EVERY == 0:
            _save_ckpt(f"{len(_done_debate)+i+1}/{len(pilot)}")

    _completed = True

except KeyboardInterrupt:
    print("\n[!] Interrupted by user — progress is saved. Re-run this cell to resume.")
except Exception as e:
    print(f"\n[X] Error: {e} — progress is saved. Fix and re-run this cell to resume.")
    raise
finally:
    _save_ckpt("final flush")
    debate_df = pd.DataFrame(debate_results)   # always defined for downstream cells

# ── Official outputs only on a clean, complete run ────────────────────────────
if _completed:
    debate_df.to_csv(OUTPUT_DIR/"debate_results.csv",index=False)
    with open(OUTPUT_DIR/"debate_transcripts.json","w") as f:
        json.dump(debate_transcripts,f,indent=2)
    print(f"\n[OK] PANDA complete — {len(debate_results)} rows saved to debate_results.csv")
else:
    print(f"\n[..] Stopped early — {len(debate_results)} images checkpointed. Re-run to resume.")


## Cell 10: Influence & Convergence Analysis

In [ ]:
influence_matrix=defaultdict(int)
stubbornness={n:0 for n in debate_agents_list}
flip_flop={n:0 for n in debate_agents_list}
change_helped={n:0 for n in debate_agents_list}
change_hurt={n:0 for n in debate_agents_list}
evidence_driven={n:0 for n in debate_agents_list}
n_img=len(debate_df)

for _,row in debate_df.iterrows():
    for n in debate_agents_list:
        ch_r2=row.get(f"r2_{n}_disease_changed",0)
        ch_r3=row.get(f"r3_{n}_disease_changed",0)
        ev_r2=row.get(f"r2_{n}_new_evidence","NONE")
        if not ch_r2 and not ch_r3: stubbornness[n]+=1
        if ch_r2 and ch_r3: flip_flop[n]+=1
        if ch_r2 and ev_r2 and ev_r2!="NONE": evidence_driven[n]+=1
        if ch_r2:
            r1c=int(row.get(f"r1_{n}_disease","")==row["gt_disease"])
            r2c=int(row.get(f"r2_{n}_disease","")==row["gt_disease"])
            if not r1c and r2c: change_helped[n]+=1
            elif r1c and not r2c: change_hurt[n]+=1
        for rnd in ["r2","r3"]:
            inf=row.get(f"{rnd}_{n}_influenced_by")
            if inf and inf!="None" and pd.notna(inf):
                for cand in debate_agents_list:
                    if cand.lower() in str(inf).lower():
                        influence_matrix[(cand,n)]+=1; break

print("Influence & Convergence:")
print(f"  {'Agent':<22} {'Stubborn%':>10} {'Flip%':>8} {'EvDriven':>9} {'Helped':>7} {'Hurt':>6}")
for n in debate_agents_list:
    print(f"  {n:<22} {stubbornness[n]/n_img:>10.1%} {flip_flop[n]/n_img:>8.1%} "
          f"{evidence_driven[n]:>9} {change_helped[n]:>7} {change_hurt[n]:>6}")
print()
print("  Influence Matrix (row=persuader, col=changed):")
print(f"  {'':22}",end="")
for n in debate_agents_list: print(f" {n:>14}",end="")
print()
for inf_n in debate_agents_list:
    print(f"  {inf_n:<22}",end="")
    for chg_n in debate_agents_list:
        v="--" if inf_n==chg_n else str(influence_matrix.get((inf_n,chg_n),0))
        print(f" {v:>14}",end="")
    print()
pd.DataFrame([{"influencer":k[0],"changed":k[1],"count":v}
              for k,v in influence_matrix.items()]).to_csv(
    OUTPUT_DIR/"influence_matrix.csv",index=False)
print("\n[OK] Influence matrix saved")

## Cell 11: Pre-Debate vs Post-Debate Accuracy

In [ ]:
base_crop=debate_df["baseline_crop_correct"].mean()
base_dis =debate_df["baseline_disease_correct"].mean()
base_both=debate_df["baseline_both_correct"].mean()
cons_crop=debate_df["consensus_crop_correct"].mean()
cons_dis =debate_df["consensus_disease_correct"].mean()
cons_both=debate_df["consensus_both_correct"].mean()
delta=cons_both-base_both

print("="*70)
print("  PANDA IMPACT: Pre-Debate vs Post-Debate Consensus")
print("="*70)
print(f"  {'':28} {'Crop':>8} {'Disease':>9} {'Combined':>10}")
print("  "+"-"*58)
print(f"  {'Pre-Debate (R1 weighted):':<28} {base_crop:>8.3f} {base_dis:>9.3f} {base_both:>10.3f}")
arr = "[^]" if delta>0.01 else ("[v]" if delta<-0.01 else "[=]")
print(f"  {'Post-Debate (R3 weighted):':<28} {cons_crop:>8.3f} {cons_dis:>9.3f} {cons_both:>10.3f}  {arr} d={delta:+.3f}")
print()
if delta>0.01:   print(f"  ++ Debate IMPROVED combined accuracy by {delta:.1%}")
elif delta<-0.01: print(f"  -- Debate HURT combined accuracy by {abs(delta):.1%}")
else:             print(f"  == No significant effect on combined accuracy")
print()
print("  PER-AGENT:")
print(f"  {'Agent':<22} {'R1 Both':>8} | {'R3 Both':>8} {'Delta':>8} {'Weight':>8}")
print("  "+"-"*58)
for n in debate_agents_list:
    r1b=debate_df.apply(lambda r:int(r.get(f"r1_{n}_crop","")==r["gt_crop"] and
                                     r.get(f"r1_{n}_disease","")==r["gt_disease"]),axis=1).mean()
    r3b=debate_df.apply(lambda r:int(r.get(f"r3_{n}_crop","")==r["gt_crop"] and
                                     r.get(f"r3_{n}_disease","")==r["gt_disease"]),axis=1).mean()
    da=r3b-r1b; arr="^" if da>0.01 else ("v" if da<-0.01 else "=")
    wt=AGENT_WEIGHTS.get(n,0)
    print(f"  {n:<22} {r1b:>8.3f} | {r3b:>8.3f} {da:>+8.3f} {arr}  w={wt:.3f}")
print(f"  {'CONSENSUS':<22} {'':>8} | {cons_both:>8.3f}")
print("="*70)

## Cell 12: PANDA Analysis Visualization

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(22,12))
fig.suptitle(f"PANDA Block — Debate Analysis  |  Agents: {', '.join(debate_agents_list)}",
             fontsize=13,fontweight="bold",y=1.01)
x=np.arange(len(debate_agents_list)); w=0.3; c2=["#8ECAE6","#023047"]

def _per(df,agents,rnd,metric):
    return [df.apply(lambda r:int(r.get(f"{rnd}_{n}_{metric}",0)==1),axis=1).mean()
            for n in agents]

r1_crop=_per(debate_df,debate_agents_list,"r1","crop_correct")
r3_crop=_per(debate_df,debate_agents_list,"r3","crop_correct")
r1_dis =_per(debate_df,debate_agents_list,"r1","disease_correct")
r3_dis =_per(debate_df,debate_agents_list,"r3","disease_correct")
r1_both=_per(debate_df,debate_agents_list,"r1","both_correct")
r3_both=_per(debate_df,debate_agents_list,"r3","both_correct")

for ax,(r1v,r3v,cv,title) in zip(
    [axes[0,0],axes[0,1],axes[1,0]],
    [(r1_crop,r3_crop,cons_crop,"Crop Accuracy: R1 vs R3"),
     (r1_dis, r3_dis, cons_dis, "Disease Accuracy: R1 vs R3"),
     (r1_both,r3_both,cons_both,"Combined Accuracy: R1 vs R3")]):
    ax.bar(x-w/2,r1v,w,label="R1 (pre-debate)", color=c2[0])
    ax.bar(x+w/2,r3v,w,label="R3 (post-debate)",color=c2[1])
    ax.axhline(cv,color="red",ls="--",lw=1.5,label=f"Consensus ({cv:.2f})")
    ax.set_xticks(x); ax.set_xticklabels(debate_agents_list,rotation=15,fontsize=9)
    ax.set_ylim(0,1.05); ax.set_title(title); ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.4)

ax=axes[0,2]
inf_mat=np.zeros((len(debate_agents_list),len(debate_agents_list)))
for i2,inf_n in enumerate(debate_agents_list):
    for j2,chg_n in enumerate(debate_agents_list):
        if i2!=j2: inf_mat[i2,j2]=influence_matrix.get((inf_n,chg_n),0)
sns.heatmap(inf_mat,annot=True,fmt=".0f",cmap="YlOrRd",ax=ax,
            xticklabels=debate_agents_list,yticklabels=debate_agents_list,
            cbar_kws={"label":"Times influenced"})
ax.set_xlabel("Agent CHANGED"); ax.set_ylabel("PERSUADER")
ax.set_title("Peer Influence Matrix"); ax.tick_params(rotation=20)

ax=axes[1,1]
stub_v=[stubbornness[n]/n_img for n in debate_agents_list]
flip_v=[flip_flop[n]/n_img    for n in debate_agents_list]
r2ch_v=[debate_df.apply(lambda r:r.get(f"r2_{n}_disease_changed",0),axis=1).mean()
        for n in debate_agents_list]
x4=np.arange(len(debate_agents_list)); w4=0.22
ax.bar(x4-w4,stub_v,w4,label="Stubborn",     color="#264653")
ax.bar(x4,   r2ch_v,w4,label="R1->R2 change",color=c2[0])
ax.bar(x4+w4,flip_v,w4,label="Flip-flop",    color="#e76f51")
ax.set_xticks(x4); ax.set_xticklabels(debate_agents_list,rotation=15,fontsize=9)
ax.set_ylim(0,1.05); ax.set_title("Agent Behavior"); ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.4)

ax=axes[1,2]
helped_v=[change_helped[n] for n in debate_agents_list]
hurt_v  =[change_hurt[n]   for n in debate_agents_list]
ax.bar(x-w/2,helped_v,w,label="Change Helped",color="#2a9d8f")
ax.bar(x+w/2,hurt_v,  w,label="Change Hurt",  color="#e63946")
ax.set_xticks(x); ax.set_xticklabels(debate_agents_list,rotation=15,fontsize=9)
ax.set_ylabel("Count"); ax.set_title("Impact of Changing Disease Verdict")
ax.legend(fontsize=8); ax.grid(axis="y",alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR/"panda_debate_analysis.png",dpi=150,bbox_inches="tight")
plt.show()
print("[OK] panda_debate_analysis.png saved")

## Cell 13: Sample Selection
Selects the same 10 images used in the VIDA sample output for direct comparison.

In [ ]:
# ── Mirror the VIDA sample selection logic exactly ────────────────────────────
# We use r1_df (loaded from VIDA) to pick the same image categories

_all_imgs = r1_df["image_name"].unique().tolist()

def _vida_n_correct(img_name):
    rows = r1_df[r1_df["image_name"]==img_name]
    return int(rows["both_correct"].sum()) if len(rows) else 0

_all_c  = [i for i in _all_imgs if _vida_n_correct(i)==len(vida_agent_names)][:3]
_all_w  = [i for i in _all_imgs if _vida_n_correct(i)==0][:3]
_split  = [i for i in _all_imgs if 0<_vida_n_correct(i)<len(vida_agent_names)][:2]
_used   = set(_all_c+_all_w+_split)
_random = [i for i in _all_imgs if i not in _used][:2]

sample_imgs = _all_c + _all_w + _split + _random

# Filter to images that have debate results
_has_debate = set(debate_df["image_name"].tolist())
sample_imgs = [i for i in sample_imgs if i in _has_debate]

# Pad with debate images if needed
if len(sample_imgs) < 9:
    extra = [i for i in _has_debate if i not in set(sample_imgs)]
    sample_imgs += extra[:9-len(sample_imgs)]

print(f"Sample images selected: {len(sample_imgs)}")
print(f"  From VIDA all-correct : {len([i for i in _all_c  if i in set(sample_imgs)])}")
print(f"  From VIDA all-wrong   : {len([i for i in _all_w  if i in set(sample_imgs)])}")
print(f"  From VIDA split-vote  : {len([i for i in _split  if i in set(sample_imgs)])}")
print(f"  Additional            : {len([i for i in _random if i in set(sample_imgs)])}")
print()
for idx, img_name in enumerate(sample_imgs, 1):
    rows   = r1_df[r1_df["image_name"]==img_name]
    gt_c   = rows["gt_crop"].values[0]   if len(rows) else "?"
    gt_d   = rows["gt_disease"].values[0] if len(rows) else "?"
    n_vida = _vida_n_correct(img_name)
    print(f"  [{idx:02d}] {img_name[:45]:<46} GT: {gt_c}/{gt_d}  VIDA: {n_vida}/{len(vida_agent_names)} correct")

## Cell 14: PANDA Sample Outputs — Debate Transcripts (Text)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PANDA SAMPLE TEXT OUTPUT — full debate transcript for each sample image
# ══════════════════════════════════════════════════════════════════════════════

SEP  = "=" * 72
THIN = "-" * 72

def _fmt_verdict(pred_crop, pred_dis, gt_crop, gt_dis):
    c = pred_crop == gt_crop if pred_crop else False
    d = pred_dis  == gt_dis  if pred_dis  else False
    if c and d: return "CORRECT"
    if c:       return "crop-only"
    return "WRONG"

print(f"PANDA BLOCK — DEBATE SAMPLE OUTPUTS  ({len(sample_imgs)} images)")
print(f"Named Peer Deliberation | Rounds 2 & 3 | Softmax-Weighted Consensus")
print()

for idx, img_name in enumerate(sample_imgs, 1):
    drow = debate_df[debate_df["image_name"]==img_name]
    if len(drow)==0: continue
    drow = drow.iloc[0]
    gt_crop    = drow["gt_crop"]
    gt_disease = drow["gt_disease"]
    base_ok    = bool(drow.get("baseline_both_correct",0))
    cons_ok    = bool(drow["consensus_both_correct"])
    cons_crop  = drow.get("consensus_crop","?")
    cons_dis   = drow.get("consensus_disease","?")
    transcripts = debate_transcripts.get(img_name, {})

    if cons_ok and not base_ok:    impact = "DEBATE HELPED"
    elif not cons_ok and base_ok:  impact = "DEBATE HURT"
    elif cons_ok and base_ok:      impact = "BOTH CORRECT"
    else:                          impact = "BOTH WRONG"

    print(SEP)
    print(f"  [{idx:02d}/{len(sample_imgs)}]  {impact}")
    print(f"  Image      : {img_name}")
    print(f"  GT         : {gt_crop}  /  {gt_disease}")
    print(f"  Baseline   : {'[OK]' if base_ok else '[X]'} R1 weighted -> {drow.get('r1_baseline_consensus_crop','?')} / {drow.get('r1_baseline_consensus_disease','?')}")
    print(f"  Consensus  : {'[OK]' if cons_ok else '[X]'} R3 weighted -> {cons_crop} / {cons_dis}")
    print(THIN)

    # R1 pre-debate
    print("  ROUND 1 — Independent Analysis (pre-debate):")
    r1_p = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,"")) for n in debate_agents_list}
    for n in debate_agents_list:
        p = r1_p[n]
        verd = _fmt_verdict(p["crop"], p["disease"], gt_crop, gt_disease)
        conf = f"conf={p['confidence']:.2f}" if p["confidence"] else "conf=N/A"
        print(f"    [{verd:10s}]  {n}")
        print(f"      Crop: {p['crop'] or 'PARSE FAIL'}  /  Disease: {p['disease'] or 'PARSE FAIL'}  ({conf})")
        rsn = str(p["reasoning"])
        print(f"      Reasoning: {rsn}")
        print()

    # R2 named debate
    print(THIN)
    print("  ROUND 2 — Named Debate:")
    for n in debate_agents_list:
        raw = transcripts.get("r2",{}).get(n,"")
        p   = parse_response(raw)
        verd = _fmt_verdict(p["crop"], p["disease"], gt_crop, gt_disease)
        conf = f"conf={p['confidence']:.2f}" if p["confidence"] else "conf=N/A"
        inf  = p["influenced_by"] or "None"
        ev   = p["new_visual_evidence"] or "NONE"
        chng = " <-- CHANGED" if (r1_p[n]["disease"]!=p["disease"] and
                                   r1_p[n]["disease"] and p["disease"]) else ""
        print(f"    [{verd:10s}]  {n}  |  Influenced by: {inf}{chng}")
        print(f"      Crop: {p['crop'] or '?'}  /  Disease: {p['disease'] or '?'}  ({conf})")
        print(f"      New visual evidence: {ev[:80]}")
        rsn = str(p["reasoning"])
        print(f"      Reasoning: {rsn}")
        print()

    # R3 final verdict
    print(THIN)
    print("  ROUND 3 — Final Verdict:")
    for n in debate_agents_list:
        raw = transcripts.get("r3",{}).get(n,"")
        p   = parse_response(raw)
        verd = _fmt_verdict(p["crop"], p["disease"], gt_crop, gt_disease)
        conf = f"conf={p['confidence']:.2f}" if p["confidence"] else "conf=N/A"
        inf  = p["influenced_by"] or "None"
        print(f"    [{verd:10s}]  {n}  |  Influenced by: {inf}")
        print(f"      Crop: {p['crop'] or '?'}  /  Disease: {p['disease'] or '?'}  ({conf})")
        rsn = str(p["reasoning"])
        print(f"      Reasoning: {rsn}")
        print()

    print(THIN)
    print(f"  CONSENSUS: {'[OK]' if cons_ok else '[X]'} {cons_crop} / {cons_dis}  |  {impact}")
    print()

print(SEP)
print(f"  Total images in PANDA : {len(debate_df)}")
print(f"  Sample shown          : {len(sample_imgs)}")
print(f"  Debate helped         : {int(((debate_df['consensus_both_correct']==1)&(debate_df['baseline_both_correct']==0)).sum())} images")
print(f"  Debate hurt           : {int(((debate_df['consensus_both_correct']==0)&(debate_df['baseline_both_correct']==1)).sum())} images")
print(f"  Pre-debate combined   : {debate_df['baseline_both_correct'].mean():.3f}")
print(f"  Post-debate combined  : {debate_df['consensus_both_correct'].mean():.3f}")
print(SEP)


## Cell 15: PANDA Sample Visualization — Paper Quality

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PANDA SAMPLE VISUALIZATION — Clean paper layout
# Layout per image:
#   Row A (thin): figure caption
#   Row B (medium): round section headers — "R1", "R2", "R3", "Consensus"
#   Row C (tall): image + agent cards + consensus box
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpecFromSubplotSpec
from PIL import Image as PILImage

plt.rcParams.update({"font.family": "DejaVu Sans"})

n_imgs   = len(sample_imgs)
n_agents = len(debate_agents_list)

# ── Colours ───────────────────────────────────────────────────────────────────
C = {
    "correct":    dict(bg="#e8f5e9", hdr="#a5d6a7", txt="#1b5e20", bdr="#2e7d32"),
    "crop_only":  dict(bg="#fff8e1", hdr="#ffe082", txt="#e65100", bdr="#f57c00"),
    "wrong":      dict(bg="#ffebee", hdr="#ef9a9a", txt="#b71c1c", bdr="#c62828"),
    "parse_fail": dict(bg="#fafafa", hdr="#e0e0e0", txt="#424242", bdr="#9e9e9e"),
}
RND_HDR = {
    "r1": dict(bg="#e3f2fd", txt="#0d47a1", label="Round 1  —  Independent Analysis"),
    "r2": dict(bg="#f3e5f5", txt="#4a148c", label="Round 2  —  Named Peer Debate"),
    "r3": dict(bg="#e8f5e9", txt="#1b5e20", label="Round 3  —  Final Verdict"),
}
IMPACT_C = {
    "Debate helped":   "#1b5e20",
    "Both correct":    "#2e7d32",
    "Debate hurt":     "#b71c1c",
    "Both incorrect":  "#7f0000",
}
SHORT = {
    "GPT-4.1":        "GPT-4.1",
    "GPT-4.1-mini":   "GPT-4.1-mini",
    "Claude-Sonnet":  "Claude Sonnet",
    "Claude-Haiku":   "Claude Haiku",
    "Grok-4.20-Fast": "Grok-4.20",
    "Gemma-3n-E4B":   "Gemma-3n",
}
PROV = {
    "GPT-4.1":        "OpenAI",
    "GPT-4.1-mini":   "OpenAI",
    "Claude-Sonnet":  "Anthropic",
    "Claude-Haiku":   "Anthropic",
    "Grok-4.20-Fast": "xAI",
    "Gemma-3n-E4B":   "Together",
}

def _pal(pc, pd, gc, gd):
    if pc is None or pd is None: return C["parse_fail"]
    if pc==gc and pd==gd: return C["correct"]
    if pc==gc:            return C["crop_only"]
    return C["wrong"]

def _wrap(t, w=22):
    words = str(t or "—").split()
    lines, cur = [], []
    for word in words:
        cur.append(word)
        if len(" ".join(cur)) > w:
            lines.append(" ".join(cur[:-1])); cur = [word]
        if len(lines) == 2: break
    if cur and len(lines) < 2: lines.append(" ".join(cur))
    return "\n".join(lines[:2])

def _excerpt(t, w=32, n=3):
    words = str(t or "").split()
    lines, cur = [], []
    for word in words:
        cur.append(word)
        if len(" ".join(cur)) > w:
            lines.append(" ".join(cur[:-1])); cur = [word]
        if len(lines) == n: break
    if cur and len(lines) < n: lines.append(" ".join(cur))
    out = "\n".join(lines[:n])
    return out + ("…" if len(words) > len(out.split()) else "")

def _silent_ax(ax):
    """Completely invisible axis — no ticks, no spines, transparent."""
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_facecolor("none")

def draw_agent_card(ax, agent, parsed, gt_crop, gt_dis, show_inf=False):
    pc = parsed.get("crop"); pd = parsed.get("disease")
    conf = parsed.get("confidence")
    inf_by = parsed.get("influenced_by") or "None"
    rsn    = parsed.get("reasoning") or ""

    pal = _pal(pc, pd, gt_crop, gt_dis)
    if pc==gt_crop and pd==gt_dis: verd = "Correct"
    elif pc is None or pd is None: verd = "Parse failure"
    elif pc==gt_crop:              verd = "Crop match"
    else:                          verd = "Incorrect"

    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor(pal["bg"])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(pal["bdr"]); sp.set_linewidth(1.0)

    # Header band
    ax.add_patch(plt.Rectangle((0,0.83),1,0.17, transform=ax.transAxes,
                                facecolor=pal["hdr"], zorder=1))
    ax.axhline(0.83, color=pal["bdr"], lw=0.7, alpha=0.5, zorder=2)

    ax.text(0.5,0.974, SHORT.get(agent,agent),
            ha="center",va="top",transform=ax.transAxes,
            fontsize=8,fontweight="bold",color=pal["txt"],zorder=3)
    ax.text(0.5,0.892, PROV.get(agent,""),
            ha="center",va="top",transform=ax.transAxes,
            fontsize=6,color=pal["txt"],alpha=0.7,fontstyle="italic",zorder=3)

    # Verdict
    ax.text(0.5,0.775, verd,
            ha="center",va="top",transform=ax.transAxes,
            fontsize=7.5,fontweight="bold",color=pal["txt"],zorder=3)
    ax.axhline(0.715, color=pal["bdr"], lw=0.4, alpha=0.3)

    # Crop
    ax.text(0.07,0.680,"Crop",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.07,0.620,_wrap(pc,22),ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#111",linespacing=1.2)

    # Disease
    ax.text(0.07,0.545,"Disease",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.07,0.483,_wrap(pd,22),ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#111",linespacing=1.2)

    ax.axhline(0.405, color="#ddd", lw=0.4)

    # Confidence
    cs = f"{conf:.2f}" if conf is not None else "N/A"
    ax.text(0.07,0.375,f"Confidence: {cs}",ha="left",va="top",
            transform=ax.transAxes,fontsize=6,color="#555")

    y = 0.315
    # Influenced By
    if show_inf:
        has = inf_by not in (None,"None","")
        ax.text(0.07,0.315,f"Influenced by:",ha="left",va="top",
                transform=ax.transAxes,fontsize=5.5,color="#999",fontstyle="italic")
        ax.text(0.07,0.262,inf_by[:22] if has else "None",ha="left",va="top",
                transform=ax.transAxes,fontsize=6,
                color=pal["txt"] if has else "#bbb",
                fontweight="bold" if has else "normal")
        y = 0.205

    ax.axhline(y-0.02, color="#ddd", lw=0.4)
    ax.text(0.07,y-0.04,"Reasoning excerpt",ha="left",va="top",
            transform=ax.transAxes,fontsize=5,color="#bbb",fontstyle="italic")
    ax.text(0.07,y-0.080,_excerpt(rsn,32,3),ha="left",va="top",
            transform=ax.transAxes,fontsize=5.8,color="#666",
            fontstyle="italic",linespacing=1.3)


def draw_consensus(ax, cons_crop, cons_dis, cons_ok, base_ok, impact):
    cc = "#1b5e20" if cons_ok else "#b71c1c"
    bg = "#e8f5e9" if cons_ok else "#ffebee"
    hd = "#a5d6a7" if cons_ok else "#ef9a9a"
    bd = "#2e7d32" if cons_ok else "#c62828"

    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor(bg)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(bd); sp.set_linewidth(1.8)

    ax.add_patch(plt.Rectangle((0,0.83),1,0.17,transform=ax.transAxes,
                                facecolor=hd,zorder=1))
    ax.axhline(0.83,color=bd,lw=0.7,alpha=0.5,zorder=2)
    ax.text(0.5,0.974,"Consensus",ha="center",va="top",transform=ax.transAxes,
            fontsize=8,fontweight="bold",color=cc,zorder=3)
    ax.text(0.5,0.892,"[OK] Correct" if cons_ok else "[X] Incorrect",
            ha="center",va="top",transform=ax.transAxes,
            fontsize=7.5,fontweight="bold",color=cc,zorder=3)

    ax.axhline(0.775,color=bd,lw=0.4,alpha=0.3)

    ax.text(0.08,0.740,"Crop",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.08,0.680,_wrap(cons_crop or "—",20),ha="left",va="top",
            transform=ax.transAxes,fontsize=7,color="#111",linespacing=1.2)

    ax.text(0.08,0.600,"Disease",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.08,0.538,_wrap(cons_dis or "—",20),ha="left",va="top",
            transform=ax.transAxes,fontsize=7,color="#111",linespacing=1.2)

    ax.axhline(0.460,color="#ddd",lw=0.4)

    ax.text(0.08,0.425,"Debate impact",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontweight="bold",fontstyle="italic")
    ic = IMPACT_C.get(impact,"#555")
    ax.text(0.08,0.368,impact.capitalize(),ha="left",va="top",
            transform=ax.transAxes,fontsize=7.5,color=ic,fontweight="bold")

    ax.axhline(0.300,color="#ddd",lw=0.4)

    ax.text(0.08,0.262,"Pre-debate vote",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontstyle="italic")
    ax.text(0.08,0.208,"Correct" if base_ok else "Incorrect",ha="left",va="top",
            transform=ax.transAxes,fontsize=6.5,
            color="#2e7d32" if base_ok else "#b71c1c",fontweight="bold")

    ax.text(0.08,0.145,"Post-debate vote",ha="left",va="top",transform=ax.transAxes,
            fontsize=5.5,color="#999",fontstyle="italic")
    ax.text(0.08,0.088,"Correct" if cons_ok else "Incorrect",ha="left",va="top",
            transform=ax.transAxes,fontsize=6.5,
            color="#2e7d32" if cons_ok else "#b71c1c",fontweight="bold")


# ── Figure ────────────────────────────────────────────────────────────────────
# Per-image rows: [caption 4%] + [round headers 8%] + [cards 88%]
# Columns: [image 1.8] + [R1 agents] + [R2 agents] + [R3 agents] + [consensus 1.3]

AGENT_W = 1.0; IMG_W = 1.8; CONS_W = 1.4
widths = [IMG_W] + [AGENT_W]*n_agents*3 + [CONS_W]

fig_w = max(24, (IMG_W + n_agents*3*AGENT_W + CONS_W) * 2.35)
fig_h = n_imgs * 6.2

fig = plt.figure(figsize=(fig_w, fig_h), facecolor="white")
outer = fig.add_gridspec(n_imgs, 1, hspace=0.15,
                          left=0.008, right=0.992,
                          top=0.968, bottom=0.025)

for row_i, img_name in enumerate(sample_imgs):
    drow = debate_df[debate_df["image_name"]==img_name]
    if len(drow)==0: continue
    drow = drow.iloc[0]

    gt_crop    = drow["gt_crop"];    gt_dis = drow["gt_disease"]
    base_ok    = bool(drow.get("baseline_both_correct",0))
    cons_ok    = bool(drow["consensus_both_correct"])
    cons_crop  = drow.get("consensus_crop","?")
    cons_dis   = drow.get("consensus_disease","?")
    transcripts = debate_transcripts.get(img_name,{})

    r1p = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,"")) for n in debate_agents_list}
    r2p = {n: parse_response(transcripts.get("r2",{}).get(n,"")) for n in debate_agents_list}
    r3p = {n: parse_response(transcripts.get("r3",{}).get(n,"")) for n in debate_agents_list}

    if   cons_ok and not base_ok: impact="Debate helped"
    elif not cons_ok and base_ok: impact="Debate hurt"
    elif cons_ok and base_ok:     impact="Both correct"
    else:                         impact="Both incorrect"
    border_c = IMPACT_C.get(impact,"#555")

    # 3-row split: [caption 4%] [headers 9%] [cards 87%]
    row_gs = GridSpecFromSubplotSpec(3,1, subplot_spec=outer[row_i],
                                     height_ratios=[0.04,0.09,0.87], hspace=0.0)

    # ── Caption strip ─────────────────────────────────────────────────────────
    ax_cap = fig.add_subplot(row_gs[0])
    ax_cap.set_facecolor("#f7f7f7")
    ax_cap.set_xlim(0,1); ax_cap.set_ylim(0,1)
    ax_cap.set_xticks([]); ax_cap.set_yticks([])
    for sp in ax_cap.spines.values(): sp.set_visible(False)
    ax_cap.axhline(0, color=border_c, lw=2.5)
    ax_cap.text(0.007,0.5,f"Figure {row_i+1}.",
                ha="left",va="center",transform=ax_cap.transAxes,
                fontsize=9.5,fontweight="bold",color="#222")
    ax_cap.text(0.052,0.5,
                f"{impact.capitalize()}   |   Ground truth:  "
                f"{gt_crop}  —  {gt_dis}",
                ha="left",va="center",transform=ax_cap.transAxes,
                fontsize=9,color="#333")

    # ── Round-header strip ─────────────────────────────────────────────────────
    # Same column widths as card row
    hdr_gs = GridSpecFromSubplotSpec(1, len(widths), subplot_spec=row_gs[1],
                                     wspace=0.012, width_ratios=widths)

    # Image header
    ax_ih = fig.add_subplot(hdr_gs[0])
    _silent_ax(ax_ih)
    ax_ih.text(0.5,0.5,"Input Image",ha="center",va="center",
               transform=ax_ih.transAxes,fontsize=9,fontweight="bold",color="#333")

    # R1 header — spans n_agents columns
    for ci in range(n_agents):
        ax_h = fig.add_subplot(hdr_gs[1+ci])
        _silent_ax(ax_h)
        if ci == n_agents//2:
            r = RND_HDR["r1"]
            ax_h.set_facecolor(r["bg"])
            for sp in ax_h.spines.values(): sp.set_visible(True); sp.set_edgecolor(r["txt"]); sp.set_linewidth(0.6)
            ax_h.text(0.5,0.5,r["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=9,fontweight="bold",color=r["txt"])
        else:
            ax_h.set_facecolor(RND_HDR["r1"]["bg"])

    # R2 header
    for ci in range(n_agents):
        ax_h = fig.add_subplot(hdr_gs[1+n_agents+ci])
        _silent_ax(ax_h)
        ax_h.set_facecolor(RND_HDR["r2"]["bg"])
        if ci == n_agents//2:
            r = RND_HDR["r2"]
            for sp in ax_h.spines.values(): sp.set_visible(True); sp.set_edgecolor(r["txt"]); sp.set_linewidth(0.6)
            ax_h.text(0.5,0.5,r["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=9,fontweight="bold",color=r["txt"])

    # R3 header
    for ci in range(n_agents):
        ax_h = fig.add_subplot(hdr_gs[1+n_agents*2+ci])
        _silent_ax(ax_h)
        ax_h.set_facecolor(RND_HDR["r3"]["bg"])
        if ci == n_agents//2:
            r = RND_HDR["r3"]
            for sp in ax_h.spines.values(): sp.set_visible(True); sp.set_edgecolor(r["txt"]); sp.set_linewidth(0.6)
            ax_h.text(0.5,0.5,r["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=9,fontweight="bold",color=r["txt"])

    # Consensus header
    ax_ch = fig.add_subplot(hdr_gs[-1])
    _silent_ax(ax_ch)
    ax_ch.set_facecolor("#f0f4f8")
    ax_ch.text(0.5,0.5,"Weighted Consensus",ha="center",va="center",
               transform=ax_ch.transAxes,fontsize=8.5,fontweight="bold",color="#37474f",
               linespacing=1.3)

    # ── Card strip ─────────────────────────────────────────────────────────────
    card_gs = GridSpecFromSubplotSpec(1, len(widths), subplot_spec=row_gs[2],
                                      wspace=0.012, width_ratios=widths)

    # Image panel
    ax_img = fig.add_subplot(card_gs[0])
    img_path_obj = next((p["image_path"] for p in pilot if p["image_name"]==img_name), None)
    if img_path_obj and Path(img_path_obj).exists():
        ax_img.imshow(np.array(PILImage.open(img_path_obj).convert("RGB")), aspect="auto")
    else:
        ax_img.set_facecolor("#f0f0f0")
        ax_img.text(0.5,0.5,"Unavailable",ha="center",va="center",
                    transform=ax_img.transAxes,fontsize=8,color="#bbb")
    ax_img.set_xticks([]); ax_img.set_yticks([])
    for sp in ax_img.spines.values():
        sp.set_visible(True); sp.set_edgecolor(border_c); sp.set_linewidth(2.5)

    # Agent name headers inside cards — shown on top of each card
    for ci, agent in enumerate(debate_agents_list):
        draw_agent_card(fig.add_subplot(card_gs[1+ci]),
                        agent, r1p[agent], gt_crop, gt_dis, show_inf=False)
        draw_agent_card(fig.add_subplot(card_gs[1+n_agents+ci]),
                        agent, r2p[agent], gt_crop, gt_dis, show_inf=True)
        draw_agent_card(fig.add_subplot(card_gs[1+n_agents*2+ci]),
                        agent, r3p[agent], gt_crop, gt_dis, show_inf=True)

    draw_consensus(fig.add_subplot(card_gs[-1]),
                   cons_crop, cons_dis, cons_ok, base_ok, impact)


# ── Title + Legend ─────────────────────────────────────────────────────────────
fig.suptitle(
    "PANDA Block  —  Named Peer Deliberation  "
    "|  CDDM Crop Disease Diagnosis Dataset",
    fontsize=13, fontweight="bold", y=0.998, color="#111")

legend_patches = [
    mpatches.Patch(fc="#e8f5e9", ec="#2e7d32", label="Correct (crop + disease)"),
    mpatches.Patch(fc="#fff8e1", ec="#f57c00", label="Crop match only"),
    mpatches.Patch(fc="#ffebee", ec="#c62828", label="Incorrect"),
    mpatches.Patch(fc="#fafafa", ec="#9e9e9e", label="Parse failure"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5,0.002),
           framealpha=1.0, edgecolor="#ccc", frameon=True)

plt.savefig(OUTPUT_DIR/"panda_sample_outputs.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("[OK] panda_sample_outputs.png saved")


## Cell 16: Summary & Output Files

In [ ]:
print("="*65)
print("  PANDA BLOCK — CDDM SUMMARY")
print("="*65)
print(f"  VIDA agents   : {vida_agent_names}")
print(f"  PANDA agents  : {debate_agents_list}")
print(f"  Images debated: {len(debate_df)}")
print()
print(f"  Pre-debate  combined acc : {debate_df['baseline_both_correct'].mean():.3f}")
print(f"  Post-debate combined acc : {debate_df['consensus_both_correct'].mean():.3f}")
print(f"  Delta                    : {delta:+.3f}")
top_p=max(debate_agents_list,
          key=lambda n:sum(influence_matrix.get((n,o),0)
                           for o in debate_agents_list if o!=n))
print(f"  Top persuader            : {top_p}")
print()
print("  Output files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"    {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size/1024:.1f} KB)")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEBATE IMPACT SHOWCASE
# Layout: image once on the left (full height) | R1 / R2 / R3 stacked on right
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpecFromSubplotSpec
from PIL import Image as PILImage

# ── Find showcase images ───────────────────────────────────────────────────────
showcase_imgs = debate_df[
    (debate_df["baseline_both_correct"] == 0) &
    (debate_df["consensus_both_correct"] == 1)
]["image_name"].tolist()
if len(showcase_imgs) < 5:
    showcase_imgs = debate_df[
        (debate_df["baseline_disease_correct"] == 0) &
        (debate_df["consensus_disease_correct"] == 1)
    ]["image_name"].tolist()
showcase_imgs = showcase_imgs[:5]

print(f"Showcase images ({len(showcase_imgs)}):")
for idx, img_name in enumerate(showcase_imgs, 1):
    row = debate_df[debate_df["image_name"]==img_name].iloc[0]
    print(f"  [{idx}] GT: {row['gt_crop']} / {row['gt_disease']}")
    print(f"       Pre : {row.get('r1_baseline_consensus_crop','?')} / {row.get('r1_baseline_consensus_disease','?')}  [X]")
    print(f"       Post: {row.get('consensus_crop','?')} / {row.get('consensus_disease','?')}  [OK]")

# ── Palette ────────────────────────────────────────────────────────────────────
C = {
    "correct":    dict(bg="#e8f5e9", hdr="#a5d6a7", txt="#1b5e20", bdr="#2e7d32"),
    "crop_only":  dict(bg="#fff8e1", hdr="#ffe082", txt="#e65100", bdr="#f57c00"),
    "wrong":      dict(bg="#ffebee", hdr="#ef9a9a", txt="#b71c1c", bdr="#c62828"),
    "parse_fail": dict(bg="#fafafa", hdr="#e0e0e0", txt="#424242", bdr="#9e9e9e"),
}
RND = {
    "r1": dict(bg="#e3f2fd", txt="#0d47a1", label="Round 1 — Independent Analysis  (all agents)"),
    "r2": dict(bg="#f3e5f5", txt="#4a148c", label="Round 2 — Named Peer Debate"),
    "r3": dict(bg="#e8f5e9", txt="#1b5e20", label="Round 3 — Final Verdict"),
}
SHORT = {
    "GPT-4.1":"GPT-4.1","GPT-4.1-mini":"GPT-4.1-mini",
    "Claude-Sonnet":"Claude Sonnet","Claude-Haiku":"Claude Haiku",
    "Grok-4.20-Fast":"Grok-4.20","Gemma-3n-E4B":"Gemma-3n",
}
PROV = {
    "GPT-4.1":"OpenAI","GPT-4.1-mini":"OpenAI",
    "Claude-Sonnet":"Anthropic","Claude-Haiku":"Anthropic",
    "Grok-4.20-Fast":"xAI","Gemma-3n-E4B":"Together",
}

vida_agents = sorted(r1_df["agent"].unique().tolist())

def _pal(pc,pd,gc,gd):
    if pc is None or pd is None: return C["parse_fail"]
    if pc==gc and pd==gd: return C["correct"]
    if pc==gc:            return C["crop_only"]
    return C["wrong"]

def _wrap(t, w=34):
    words = str(t or "—").split(); lines, cur = [], []
    for word in words:
        cur.append(word)
        if len(" ".join(cur)) > w: lines.append(" ".join(cur[:-1])); cur = [word]
        if len(lines) == 2: break
    if cur and len(lines) < 2: lines.append(" ".join(cur))
    return "\n".join(lines[:2])

def _excerpt(t, w=30, n=99):
    words = str(t or "").split(); lines, cur = [], []
    for word in words:
        cur.append(word)
        if len(" ".join(cur)) > w: lines.append(" ".join(cur[:-1])); cur = [word]
        if len(lines) == n: break
    if cur and len(lines) < n: lines.append(" ".join(cur))
    return "\n".join(lines)

def _silent(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_facecolor("none")

def draw_card(ax, agent, parsed, gt_crop, gt_dis, show_inf=False):
    pc = parsed.get("crop"); pd = parsed.get("disease")
    conf = parsed.get("confidence"); rsn = parsed.get("reasoning") or ""
    inf_by = parsed.get("influenced_by") or "None"
    pal = _pal(pc, pd, gt_crop, gt_dis)
    if pc==gt_crop and pd==gt_dis:   verd = "Correct"
    elif pc is None or pd is None:   verd = "Parse failure"
    elif pc==gt_crop:                verd = "Crop match"
    else:                            verd = "Incorrect"

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor(pal["bg"])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(pal["bdr"]); sp.set_linewidth(0.9)

    # Header band
    ax.add_patch(plt.Rectangle((0,0.83),1,0.17,transform=ax.transAxes,
                                facecolor=pal["hdr"],zorder=1))
    ax.axhline(0.83, color=pal["bdr"], lw=0.6, alpha=0.4, zorder=2)
    ax.text(0.5,0.975, SHORT.get(agent,agent),
            ha="center",va="top",transform=ax.transAxes,
            fontsize=9.5,fontweight="bold",color=pal["txt"],zorder=3)
    ax.text(0.5,0.893, PROV.get(agent,""),
            ha="center",va="top",transform=ax.transAxes,
            fontsize=7.5,color=pal["txt"],alpha=0.7,fontstyle="italic",zorder=3)

    ax.text(0.5,0.775, verd,
            ha="center",va="top",transform=ax.transAxes,
            fontsize=9,fontweight="bold",color=pal["txt"])
    ax.axhline(0.715, color=pal["bdr"], lw=0.4, alpha=0.25)

    ax.text(0.06,0.678,"Crop",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.06,0.618,_wrap(pc,20),ha="left",va="top",transform=ax.transAxes,
            fontsize=8.5,color="#111",linespacing=1.3)

    ax.text(0.06,0.545,"Disease",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.06,0.483,_wrap(pd,20),ha="left",va="top",transform=ax.transAxes,
            fontsize=8.5,color="#111",linespacing=1.3)

    ax.axhline(0.408, color="#ddd", lw=0.4)
    cs = f"{conf:.2f}" if conf is not None else "N/A"
    ax.text(0.06,0.380,f"Confidence: {cs}",ha="left",va="top",
            transform=ax.transAxes,fontsize=7.5,color="#555")

    # Influenced by
    if show_inf:
        has = inf_by not in (None,"None","")
        ax.text(0.06,0.318,"Influenced by:",ha="left",va="top",
                transform=ax.transAxes,fontsize=7,color="#999",fontstyle="italic")
        ax.text(0.06,0.262,inf_by[:24] if has else "None",ha="left",va="top",
                transform=ax.transAxes,fontsize=7.5,
                color=pal["txt"] if has else "#bbb",
                fontweight="bold" if has else "normal")
        rsn_top = 0.195
    else:
        rsn_top = 0.315

    ax.axhline(rsn_top-0.012, color="#ddd", lw=0.4)
    ax.text(0.06, rsn_top-0.032, "Reasoning",
            ha="left",va="top",transform=ax.transAxes,
            fontsize=6.5,color="#bbb",fontstyle="italic")

    # Reasoning — wrap at 30 chars, cap at lines that fit within remaining space
    # Available space below reasoning label = rsn_top-0.05 down to 0.01
    available = rsn_top - 0.06  # fraction of axes height available
    line_height = 0.052          # each line takes ~5.2% of axes height at fontsize 7
    max_lines = max(2, int(available / line_height))

    words = rsn.split(); lines_out, cur = [], []
    for word in words:
        cur.append(word)
        if len(" ".join(cur)) > 30:
            lines_out.append(" ".join(cur[:-1])); cur = [word]
        if len(lines_out) == max_lines: break
    if cur and len(lines_out) < max_lines:
        lines_out.append(" ".join(cur))
    rsn_text = "\n".join(lines_out[:max_lines])
    if len(words) > len(rsn_text.split()): rsn_text += " ..."

    ax.text(0.06, rsn_top-0.060, rsn_text,
            ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#444",linespacing=1.45)


def draw_consensus(ax, cons_crop, cons_dis, cons_ok, base_ok):
    cc="#1b5e20" if cons_ok else "#b71c1c"
    bg="#e8f5e9" if cons_ok else "#ffebee"
    hd="#a5d6a7" if cons_ok else "#ef9a9a"
    bd="#2e7d32" if cons_ok else "#c62828"
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor(bg)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor(bd); sp.set_linewidth(1.8)
    ax.add_patch(plt.Rectangle((0,0.83),1,0.17,transform=ax.transAxes,facecolor=hd,zorder=1))
    ax.axhline(0.83,color=bd,lw=0.6,alpha=0.4,zorder=2)
    ax.text(0.5,0.975,"Consensus",ha="center",va="top",transform=ax.transAxes,
            fontsize=9.5,fontweight="bold",color=cc,zorder=3)
    ax.text(0.5,0.893,"[OK] Correct" if cons_ok else "[X] Incorrect",
            ha="center",va="top",transform=ax.transAxes,fontsize=8.5,fontweight="bold",color=cc,zorder=3)
    ax.axhline(0.775,color=bd,lw=0.4,alpha=0.3)
    ax.text(0.08,0.738,"Crop",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.08,0.678,_wrap(cons_crop or "—",18),ha="left",va="top",
            transform=ax.transAxes,fontsize=8.5,color="#111",linespacing=1.3)
    ax.text(0.08,0.598,"Disease",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontweight="bold",fontstyle="italic")
    ax.text(0.08,0.536,_wrap(cons_dis or "—",18),ha="left",va="top",
            transform=ax.transAxes,fontsize=8.5,color="#111",linespacing=1.3)
    ax.axhline(0.458,color="#ddd",lw=0.4)
    ax.text(0.08,0.420,"Pre-debate vote",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontstyle="italic")
    ax.text(0.08,0.365,"Correct" if base_ok else "Incorrect",ha="left",va="top",
            transform=ax.transAxes,fontsize=7.5,
            color="#2e7d32" if base_ok else "#b71c1c",fontweight="bold")
    ax.text(0.08,0.285,"Post-debate vote",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontstyle="italic")
    ax.text(0.08,0.228,"Correct" if cons_ok else "Incorrect",ha="left",va="top",
            transform=ax.transAxes,fontsize=7.5,
            color="#2e7d32" if cons_ok else "#b71c1c",fontweight="bold")
    ax.text(0.08,0.135,"Debate impact",ha="left",va="top",transform=ax.transAxes,
            fontsize=7,color="#999",fontstyle="italic")
    ax.text(0.08,0.072,"Debate helped",ha="left",va="top",
            transform=ax.transAxes,fontsize=8,color="#1b5e20",fontweight="bold")

# ── Figure layout ──────────────────────────────────────────────────────────────
# Per showcase image:
#   [caption strip]
#   [image (full height) | rounds stacked vertically]
#
# Right side stacked:
#   [R1 header + cards]
#   [thin divider]
#   [R2 header + cards]
#   [thin divider]
#   [R3 header + cards + consensus]

n_vida   = len(vida_agents)
n_debate = len(debate_agents_list)
n_show   = len(showcase_imgs)

AGENT_W = 1.0; IMG_W = 1.4; CONS_W = 1.3

# Each round row height
RND_H = 1.0   # relative height per round
DIV_H = 0.06  # divider

fig_w = max(26, (IMG_W + max(n_vida, n_debate)*AGENT_W + CONS_W) * 2.0)
fig_h = n_show * 30.0

fig = plt.figure(figsize=(fig_w, fig_h), facecolor="white")
outer = fig.add_gridspec(n_show, 1, hspace=0.08,
                          left=0.006, right=0.994,
                          top=0.970, bottom=0.012)

for row_i, img_name in enumerate(showcase_imgs):
    drow = debate_df[debate_df["image_name"]==img_name]
    if len(drow)==0: continue
    drow = drow.iloc[0]
    gt_crop = drow["gt_crop"]; gt_dis = drow["gt_disease"]
    base_ok = bool(drow.get("baseline_both_correct",0))
    cons_ok = bool(drow["consensus_both_correct"])
    cons_crop = drow.get("consensus_crop","?"); cons_dis = drow.get("consensus_disease","?")
    transcripts = debate_transcripts.get(img_name,{})

    r1p_all    = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,"")) for n in vida_agents}
    r1p_debate = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,"")) for n in debate_agents_list}
    r2p = {n: parse_response(transcripts.get("r2",{}).get(n,"")) for n in debate_agents_list}
    r3p = {n: parse_response(transcripts.get("r3",{}).get(n,"")) for n in debate_agents_list}

    # Each showcase block:
    # row 0: caption (2%)
    # row 1: content (98%) split into [image | rounds]
    block_gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[row_i],
                                       height_ratios=[0.025, 0.975], hspace=0.0)

    # Caption
    ax_cap = fig.add_subplot(block_gs[0])
    ax_cap.set_facecolor("#f5f5f5"); ax_cap.set_xlim(0,1); ax_cap.set_ylim(0,1)
    ax_cap.set_xticks([]); ax_cap.set_yticks([])
    for sp in ax_cap.spines.values(): sp.set_visible(False)
    ax_cap.axhline(0, color="#1b5e20", lw=3.0)
    ax_cap.text(0.006,0.5,f"Showcase {row_i+1}.",
                ha="left",va="center",transform=ax_cap.transAxes,
                fontsize=11,fontweight="bold",color="#222")
    ax_cap.text(0.060,0.5,
                f"Debate corrected the consensus   |   "
                f"Ground truth:  {gt_crop}  —  {gt_dis}   |   "
                f"Pre-debate: [X]   Post-debate: [OK]",
                ha="left",va="center",transform=ax_cap.transAxes,
                fontsize=10.5,color="#333")

    # Content: [image | right_panel]
    # Right panel = R1 rounds + divider + R2 + divider + R3
    # image width ratio vs total agents
    # Image takes IMG_W, right panel takes the rest — no fixed column count needed
    content_gs = GridSpecFromSubplotSpec(1, 2, subplot_spec=block_gs[1],
                                         wspace=0.012,
                                         width_ratios=[IMG_W, n_vida * AGENT_W])

    # ── Image (spans full height) ─────────────────────────────────────────────
    ax_img = fig.add_subplot(content_gs[0])
    img_path_obj = next((p["image_path"] for p in pilot if p["image_name"]==img_name), None)
    if img_path_obj and Path(img_path_obj).exists():
        ax_img.imshow(np.array(PILImage.open(img_path_obj).convert("RGB")), aspect="auto")
    else:
        ax_img.set_facecolor("#f0f0f0")
        ax_img.text(0.5,0.5,"Unavailable",ha="center",va="center",
                    transform=ax_img.transAxes,fontsize=9,color="#bbb")
    ax_img.set_xticks([]); ax_img.set_yticks([])
    for sp in ax_img.spines.values():
        sp.set_visible(True); sp.set_edgecolor("#1b5e20"); sp.set_linewidth(2.5)

    # GT label at bottom of image
    ax_img.text(0.5, 0.015,
                f"GT: {gt_crop} — {gt_dis}",
                ha="center", va="bottom", transform=ax_img.transAxes,
                fontsize=8, color="white", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#00000088", edgecolor="none"))

    # ── Right panel: R1 / divider / R2 / divider / R3 ─────────────────────────
    # heights: R1=33%, div=1%, R2=30%, div=1%, R3=35%
    right_gs = GridSpecFromSubplotSpec(5, 1, subplot_spec=content_gs[1],
                                       height_ratios=[0.330, 0.007, 0.320, 0.007, 0.336],
                                       hspace=0.0)

    # ── R1 ────────────────────────────────────────────────────────────────────
    r1_gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=right_gs[0],
                                    height_ratios=[0.095, 0.905], hspace=0.0)
    # R1 header
    r1_hdr_gs = GridSpecFromSubplotSpec(1, n_vida, subplot_spec=r1_gs[0],
                                            wspace=0.01,
                                            width_ratios=[1]*n_vida)
    for ci in range(n_vida):
        ax_h = fig.add_subplot(r1_hdr_gs[ci]); _silent(ax_h)
        ax_h.set_facecolor(RND["r1"]["bg"])
        for sp in ax_h.spines.values():
            sp.set_visible(True); sp.set_edgecolor(RND["r1"]["txt"]); sp.set_linewidth(0.6)
        if ci == n_vida//2:
            ax_h.text(0.5,0.5,RND["r1"]["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=10.5,fontweight="bold",
                      color=RND["r1"]["txt"])
    # R1 cards
    r1_card_gs = GridSpecFromSubplotSpec(1, n_vida, subplot_spec=r1_gs[1],
                                             wspace=0.012,
                                             width_ratios=[1]*n_vida)
    for ci, agent in enumerate(vida_agents):
        draw_card(fig.add_subplot(r1_card_gs[ci]),
                  agent, r1p_all[agent], gt_crop, gt_dis, show_inf=False)

    # ── Divider 1 ─────────────────────────────────────────────────────────────
    ax_div1 = fig.add_subplot(right_gs[1]); _silent(ax_div1)
    ax_div1.set_facecolor("white")
    ax_div1.axhline(0.5, color="#bbb", lw=0.8, linestyle="--")

    # ── R2 ────────────────────────────────────────────────────────────────────
    r2_gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=right_gs[2],
                                    height_ratios=[0.095, 0.905], hspace=0.0)
    r2_hdr_gs = GridSpecFromSubplotSpec(1, n_debate, subplot_spec=r2_gs[0],
                                            wspace=0.01,
                                            width_ratios=[1]*n_debate)
    for ci in range(n_debate):
        ax_h = fig.add_subplot(r2_hdr_gs[ci]); _silent(ax_h)
        ax_h.set_facecolor(RND["r2"]["bg"])
        for sp in ax_h.spines.values():
            sp.set_visible(True); sp.set_edgecolor(RND["r2"]["txt"]); sp.set_linewidth(0.6)
        if ci == n_debate//2:
            ax_h.text(0.5,0.5,RND["r2"]["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=10.5,fontweight="bold",
                      color=RND["r2"]["txt"])
    r2_card_gs = GridSpecFromSubplotSpec(1, n_debate, subplot_spec=r2_gs[1],
                                             wspace=0.012,
                                             width_ratios=[1]*n_debate)
    for ci, agent in enumerate(debate_agents_list):
        draw_card(fig.add_subplot(r2_card_gs[ci]),
                  agent, r2p[agent], gt_crop, gt_dis, show_inf=True)

    # ── Divider 2 ─────────────────────────────────────────────────────────────
    ax_div2 = fig.add_subplot(right_gs[3]); _silent(ax_div2)
    ax_div2.set_facecolor("white")
    ax_div2.axhline(0.5, color="#bbb", lw=0.8, linestyle="--")

    # ── R3 + Consensus ────────────────────────────────────────────────────────
    # R3 fills full width: n_debate equal cols + consensus
    r3_agent_w = n_debate / (n_debate + CONS_W/AGENT_W)
    r3_cons_w  = (CONS_W/AGENT_W) / (n_debate + CONS_W/AGENT_W)
    r3_widths = [1]*n_debate + [CONS_W/AGENT_W]
    r3_gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=right_gs[4],
                                    height_ratios=[0.085, 0.915], hspace=0.0)
    r3_hdr_gs = GridSpecFromSubplotSpec(1, n_debate+1, subplot_spec=r3_gs[0],
                                        wspace=0.01, width_ratios=r3_widths)
    for ci in range(n_debate):
        ax_h = fig.add_subplot(r3_hdr_gs[ci]); _silent(ax_h)
        ax_h.set_facecolor(RND["r3"]["bg"])
        for sp in ax_h.spines.values():
            sp.set_visible(True); sp.set_edgecolor(RND["r3"]["txt"]); sp.set_linewidth(0.6)
        if ci == n_debate//2:
            ax_h.text(0.5,0.5,RND["r3"]["label"],ha="center",va="center",
                      transform=ax_h.transAxes,fontsize=10.5,fontweight="bold",
                      color=RND["r3"]["txt"])
    ax_ch = fig.add_subplot(r3_hdr_gs[-1]); _silent(ax_ch)
    ax_ch.set_facecolor("#eceff1")
    ax_ch.text(0.5,0.5,"Weighted Consensus",ha="center",va="center",
               transform=ax_ch.transAxes,fontsize=10,fontweight="bold",color="#37474f")

    r3_card_gs = GridSpecFromSubplotSpec(1, n_debate+1, subplot_spec=r3_gs[1],
                                         wspace=0.012, width_ratios=r3_widths)
    for ci, agent in enumerate(debate_agents_list):
        draw_card(fig.add_subplot(r3_card_gs[ci]),
                  agent, r3p[agent], gt_crop, gt_dis, show_inf=True)
    draw_consensus(fig.add_subplot(r3_card_gs[-1]),
                   cons_crop, cons_dis, cons_ok, base_ok)


# ── Title + Legend ─────────────────────────────────────────────────────────────
fig.suptitle(
    "Debate Impact Showcase  —  "
    "5 Images Where Named Peer Deliberation Corrected the Consensus Answer",
    fontsize=13, fontweight="bold", y=0.997, color="#111")

legend_patches = [
    mpatches.Patch(fc="#e8f5e9",ec="#2e7d32",label="Correct (crop + disease)"),
    mpatches.Patch(fc="#fff8e1",ec="#f57c00",label="Crop match only"),
    mpatches.Patch(fc="#ffebee",ec="#c62828",label="Incorrect"),
    mpatches.Patch(fc="#fafafa",ec="#9e9e9e",label="Parse failure"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5,0.004),
           framealpha=1.0, edgecolor="#ccc", frameon=True)

plt.savefig(OUTPUT_DIR/"debate_impact_showcase.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("[OK] debate_impact_showcase.png saved")


In [ ]:
print("="*65)
print("  PANDA BLOCK — CDDM SUMMARY")
print("="*65)
print(f"  VIDA agents   : {vida_agent_names}")
print(f"  PANDA agents  : {debate_agents_list}")
print(f"  Images debated: {len(debate_df)}")
print()
print(f"  Pre-debate  combined acc : {debate_df['baseline_both_correct'].mean():.3f}")
print(f"  Post-debate combined acc : {debate_df['consensus_both_correct'].mean():.3f}")
print(f"  Delta                    : {delta:+.3f}")
top_p=max(debate_agents_list,
          key=lambda n:sum(influence_matrix.get((n,o),0)
                           for o in debate_agents_list if o!=n))
print(f"  Top persuader            : {top_p}")
print()
print("  Output files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"    {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size/1024:.1f} KB)")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEMANTIC EVALUATION — 3 GPT-5 Judge Metrics Across Debate Rounds
#
# Metric 1 — Semantic Label Similarity (0–10)
#   How semantically close is the predicted crop+disease to ground truth?
#   Captures partial credit: "Leaf Blight" vs "Bacterial Leaf Blight" scores ~7
#   vs exact-match which scores 0.
#
# Metric 2 — Reasoning Specificity (0–10)
#   Does the reasoning cite specific visual features (lesion color, shape,
#   texture, pattern, margins)? Measures visual grounding quality.
#   Hypothesis: debate forces agents to be more specific R1→R3.
#
# Metric 3 — Inter-Agent Reasoning Convergence (0–10)
#   Do the debate agents reference the same visual evidence in their reasoning?
#   Measures epistemic alignment across agents independent of label accuracy.
#   Novel metric: shows debate produces shared visual understanding.
# ══════════════════════════════════════════════════════════════════════════════

import time
import random

# ── Sampling strategy ─────────────────────────────────────────────────────────
# Running all 490 images x 3 agents x 3 rounds x 3 metrics = expensive
# Sample N_EVAL images stratified: correct / wrong / split consensus
# Full dataset evaluation. NOTE: GPT-5 is a reasoning model — hidden reasoning
# tokens make each judge call slower/pricier than GPT-4.1. Budget accordingly.
random.seed(42)

# Use all images that have debate transcripts
eval_imgs = debate_df["image_name"].tolist()

n_correct_imgs = debate_df["consensus_both_correct"].sum()
n_wrong_imgs   = len(debate_df) - n_correct_imgs
n_calls = len(eval_imgs) * len(debate_agents_list) * 2 + len(eval_imgs) * 3
print(f"Semantic evaluation on ALL {len(eval_imgs)} images")
print(f"  Consensus correct : {int(n_correct_imgs)}")
print(f"  Consensus wrong   : {int(n_wrong_imgs)}")
print(f"  Agents evaluated  : {debate_agents_list}")
print(f"  Rounds            : R1, R2, R3")
print(f"  Metrics per call  : 3 (label sim, specificity, convergence)")
print(f"  Total API calls   : ~{n_calls:,}")
print(f"  Judge model       : {JUDGE_MODEL} (non-participant — removes self-bias, reviewer R1.3)")
print(f"  NOTE              : GPT-5 reasoning tokens add latency + cost vs GPT-4.1")
print()

# ── Robust judge wrapper (default GPT-5, a reasoning model) ───────────────────
# GPT-5 spends tokens on HIDDEN reasoning first. With a small budget it returns an
# EMPTY string (finish_reason='length'). So: use max_completion_tokens (NOT
# max_tokens), reasoning_effort='minimal', a generous budget, and grow-and-retry on
# empty output. Falls back to max_tokens if JUDGE_MODEL is reset to 'gpt-4.1'.
def _judge_call(prompt, max_out=2000, retries=4):
    is_gpt5 = str(JUDGE_MODEL).startswith("gpt-5")
    for attempt in range(retries):
        try:
            kwargs = {"model": JUDGE_MODEL,
                      "messages": [{"role": "user", "content": prompt}]}
            if is_gpt5:
                kwargs["max_completion_tokens"] = max_out
                kwargs["reasoning_effort"] = "minimal"   # GPT-5 does NOT accept temperature
            else:
                kwargs["max_tokens"] = max_out
            resp = openai_client.chat.completions.create(**kwargs)
            txt = (resp.choices[0].message.content or "").strip()
            if not txt and resp.choices[0].finish_reason == "length":
                max_out = min(max_out * 2, 8000)   # reasoning ate the budget — grow & retry
                continue
            if txt:
                return txt
        except Exception:
            if attempt < retries - 1: time.sleep(3)
    return ""

def _extract_score(text):
    m = re.findall(r"[0-9]+\.?[0-9]*", text or "")
    return min(10.0, max(0.0, float(m[0]))) if m else None

def judge_semantic_label(gt_crop, gt_disease, pred_crop, pred_disease, retries=3):
    """Metric 1: Semantic Label Similarity (0-10)."""
    if not pred_crop and not pred_disease:
        return None
    prompt = (
        f"You are a plant pathology expert evaluating a crop disease diagnosis prediction.\n\n"
        f"Ground truth: Crop={gt_crop}, Disease={gt_disease}\n"
        f"Prediction  : Crop={pred_crop or 'unknown'}, Disease={pred_disease or 'unknown'}\n\n"
        f"Score the semantic similarity of the PREDICTION to the GROUND TRUTH on a 0-10 scale:\n"
        f"  10 = Exact match (same crop, same disease)\n"
        f"   8 = Correct crop, disease is same category (e.g. 'Leaf Blight' vs 'Bacterial Leaf Blight')\n"
        f"   6 = Correct crop, disease is related (same pathogen type or symptom class)\n"
        f"   4 = Correct crop only, disease is unrelated\n"
        f"   2 = Wrong crop but disease type is plausible for the actual crop\n"
        f"   0 = Completely wrong (wrong crop, unrelated disease)\n\n"
        f"Respond with ONLY a single number 0-10. No explanation."
    )
    return _extract_score(_judge_call(prompt))

def judge_reasoning_specificity(gt_crop, gt_disease, reasoning, retries=3):
    """Metric 2: Reasoning Specificity (0-10)."""
    if not reasoning or len(reasoning.strip()) < 10:
        return None
    prompt = (
        f"You are a plant pathology expert evaluating the quality of a disease diagnosis reasoning.\n\n"
        f"Ground truth: {gt_crop} with {gt_disease}\n"
        f"Reasoning: {reasoning[:400]}\n\n"
        f"Score the VISUAL SPECIFICITY of this reasoning on a 0-10 scale:\n"
        f"  10 = Cites 3+ specific visual features with precise detail "
        f"(e.g. 'tan lesions with dark concentric rings at leaf margins', "
        f"'chlorotic halo surrounding necrotic spots', 'water-soaked appearance')\n"
        f"   7 = Cites 2 specific visual features with some detail\n"
        f"   5 = Mentions visual features but vaguely (e.g. 'brown spots', 'yellowing')\n"
        f"   3 = Mostly generic statements with minimal visual evidence\n"
        f"   0 = No visual features cited, only label repetition\n\n"
        f"Respond with ONLY a single number 0-10. No explanation."
    )
    return _extract_score(_judge_call(prompt))

def judge_convergence(gt_crop, gt_disease, reasonings_dict, retries=3):
    """Metric 3: Inter-Agent Reasoning Convergence (0-10)."""
    valid = {n: r for n, r in reasonings_dict.items() if r and len(r.strip()) > 10}
    if len(valid) < 2:
        return None
    agents_text = "\n\n".join(f"[{name}]: {rsn[:250]}" for name, rsn in valid.items())
    prompt = (
        f"You are a plant pathology expert.\n\n"
        f"Ground truth: {gt_crop} with {gt_disease}\n\n"
        f"Below are reasoning statements from {len(valid)} AI agents analyzing the SAME leaf image:\n\n"
        f"{agents_text}\n\n"
        f"Score the VISUAL EVIDENCE CONVERGENCE across these agents on a 0-10 scale:\n"
        f"  10 = All agents describe the same specific visual features "
        f"(same lesion type, color, location, pattern)\n"
        f"   7 = Agents agree on 2+ visual features, minor differences in detail\n"
        f"   5 = Partial overlap — some shared features, some contradictions\n"
        f"   3 = Agents describe different visual features, little overlap\n"
        f"   0 = Completely contradictory — agents see entirely different things\n\n"
        f"Respond with ONLY a single number 0-10. No explanation."
    )
    return _extract_score(_judge_call(prompt))


# ── Run evaluation ────────────────────────────────────────────────────────────
print("Running GPT-4.1 semantic evaluation...")
print("(This will take ~10-20 min depending on API speed)\n")

sem_results = []

for img_idx, img_name in enumerate(tqdm(eval_imgs, desc="Semantic eval")):
    drow = debate_df[debate_df["image_name"]==img_name]
    if len(drow)==0: continue
    drow = drow.iloc[0]
    gt_crop    = drow["gt_crop"]
    gt_disease = drow["gt_disease"]
    transcripts = debate_transcripts.get(img_name, {})
    cons_ok = bool(drow["consensus_both_correct"])

    # Parse all rounds for debate agents
    r1p = {n: parse_response(r1_raw_bank.get(img_name,{}).get(n,"")) for n in debate_agents_list}
    r2p = {n: parse_response(transcripts.get("r2",{}).get(n,"")) for n in debate_agents_list}
    r3p = {n: parse_response(transcripts.get("r3",{}).get(n,"")) for n in debate_agents_list}

    round_data = [("r1", r1p), ("r2", r2p), ("r3", r3p)]

    # Metric 3: convergence per round (one score per round, covers all agents)
    for rnd_key, rnd_parsed in round_data:
        reasonings = {n: rnd_parsed[n].get("reasoning","") for n in debate_agents_list}
        conv_score = judge_convergence(gt_crop, gt_disease, reasonings)
        sem_results.append({
            "image_name": img_name,
            "gt_crop": gt_crop,
            "gt_disease": gt_disease,
            "consensus_ok": cons_ok,
            "agent": "__convergence__",
            "round": rnd_key,
            "metric": "convergence",
            "score": conv_score,
        })

    # Metrics 1 & 2: per agent per round
    for n in debate_agents_list:
        for rnd_key, rnd_parsed in round_data:
            p = rnd_parsed[n]
            pred_crop = p.get("crop"); pred_dis = p.get("disease")
            reasoning = p.get("reasoning","")

            # Metric 1: semantic label similarity
            sim = judge_semantic_label(gt_crop, gt_disease, pred_crop, pred_dis)
            sem_results.append({
                "image_name": img_name,
                "gt_crop": gt_crop,
                "gt_disease": gt_disease,
                "consensus_ok": cons_ok,
                "agent": n,
                "round": rnd_key,
                "metric": "label_similarity",
                "score": sim,
            })

            # Metric 2: reasoning specificity
            spec = judge_reasoning_specificity(gt_crop, gt_disease, reasoning)
            sem_results.append({
                "image_name": img_name,
                "gt_crop": gt_crop,
                "gt_disease": gt_disease,
                "consensus_ok": cons_ok,
                "agent": n,
                "round": rnd_key,
                "metric": "specificity",
                "score": spec,
            })

    # Checkpoint every 10 images
    if (img_idx+1) % 10 == 0:
        pd.DataFrame(sem_results).to_csv(OUTPUT_DIR/"semantic_eval_checkpoint.csv", index=False)

sem_df = pd.DataFrame(sem_results)
sem_df.to_csv(OUTPUT_DIR/"semantic_eval_results.csv", index=False)
print(f"\n[OK] Semantic evaluation complete — {len(sem_df)} scores saved")
print(sem_df.groupby(["metric","round"])["score"].mean().round(3).to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEMANTIC EVALUATION VISUALIZATION
# 3 rows: one per metric
# Each row: per-round mean with per-agent breakdown + trajectory line
# ══════════════════════════════════════════════════════════════════════════════

# Load from checkpoint if re-running
if "sem_df" not in dir() or len(sem_df) == 0:
    _ckpt = OUTPUT_DIR/"semantic_eval_results.csv"
    if _ckpt.exists():
        sem_df = pd.read_csv(_ckpt)
        print(f"[OK] Loaded {len(sem_df)} scores from checkpoint")
    else:
        raise FileNotFoundError("Run Cell 18 first to generate semantic_eval_results.csv")

ROUND_ORDER  = ["r1","r2","r3"]
ROUND_LABELS = {"r1":"R1\n(Independent)","r2":"R2\n(Named Debate)","r3":"R3\n(Final Verdict)"}
AGENT_COLORS = {
    debate_agents_list[0]: "#1565c0",
    debate_agents_list[1]: "#6a1b9a",
    debate_agents_list[2]: "#2e7d32",
}
METRIC_META = {
    "label_similarity": dict(
        title="Metric 1 — Semantic Label Similarity",
        ylabel="Score (0–10)",
        description=(
            "Semantic closeness of predicted crop+disease to ground truth.\n"
            "Partial credit: 'Leaf Blight' vs 'Bacterial Leaf Blight' → ~8  |  "
            "exact match → 10  |  wrong crop → 0–2"
        ),
        color="#1565c0",
    ),
    "specificity": dict(
        title="Metric 2 — Reasoning Specificity",
        ylabel="Score (0–10)",
        description=(
            "Degree to which reasoning cites specific visual features\n"
            "(lesion color, shape, texture, pattern, margins).\n"
            "Hypothesis: named debate forces agents to be more visually grounded R1→R3."
        ),
        color="#6a1b9a",
    ),
    "convergence": dict(
        title="Metric 3 — Inter-Agent Reasoning Convergence",
        ylabel="Score (0–10)",
        description=(
            "Do agents reference the same visual evidence in their reasoning?\n"
            "Measures epistemic alignment across agents, independent of label accuracy.\n"
            "Novel: shows debate produces shared visual understanding."
        ),
        color="#2e7d32",
    ),
}

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.patch.set_facecolor("white")
fig.suptitle(
    "Semantic Evaluation: GPT-4.1 Judge Scores Across Debate Rounds\n"
    f"PANDA Block  |  Agents: {', '.join(debate_agents_list)}  |  "
    f"N={len(sem_df['image_name'].unique())} images (full dataset)",
    fontsize=13, fontweight="bold", y=1.01, color="#111"
)

for row_i, metric in enumerate(["label_similarity","specificity","convergence"]):
    meta = METRIC_META[metric]
    sub = sem_df[sem_df["metric"]==metric].dropna(subset=["score"])

    # ── Col 0: Mean per round + trajectory ───────────────────────────────────
    ax = axes[row_i, 0]
    ax.set_facecolor("#fafafa")
    for sp in ax.spines.values(): sp.set_color("#ddd")

    round_means = sub.groupby("round")["score"].mean().reindex(ROUND_ORDER)
    round_sems  = sub.groupby("round")["score"].sem().reindex(ROUND_ORDER)
    x = np.arange(3)

    bars = ax.bar(x, round_means.values, color=meta["color"], alpha=0.75,
                  width=0.5, zorder=2)
    ax.errorbar(x, round_means.values, yerr=round_sems.values*1.96,
                fmt="none", color="#333", capsize=5, lw=1.5, zorder=3)
    ax.plot(x, round_means.values, "o-", color="#111", lw=2, ms=7, zorder=4)

    for bar, val in zip(bars, round_means.values):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.08,
                f"{val:.2f}", ha="center", fontsize=9.5, fontweight="bold", color="#111")

    ax.set_xticks(x)
    ax.set_xticklabels([ROUND_LABELS[r] for r in ROUND_ORDER], fontsize=10)
    ax.set_ylim(0, 11)
    ax.set_ylabel(meta["ylabel"], fontsize=10)
    ax.set_title(meta["title"], fontsize=11, fontweight="bold", pad=8)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    ax.set_xlabel(meta["description"], fontsize=7.5, color="#555", labelpad=8)

    # Delta annotations
    if len(round_means.dropna()) == 3:
        d12 = round_means["r2"] - round_means["r1"]
        d23 = round_means["r3"] - round_means["r2"]
        dtot = round_means["r3"] - round_means["r1"]
        ax.annotate("", xy=(1, round_means["r2"]), xytext=(0, round_means["r1"]),
                    arrowprops=dict(arrowstyle="->", color="#e65100", lw=1.5))
        ax.annotate("", xy=(2, round_means["r3"]), xytext=(1, round_means["r2"]),
                    arrowprops=dict(arrowstyle="->", color="#e65100", lw=1.5))
        ax.text(2.35, 10.3, f"Total Δ = {dtot:+.2f}",
                fontsize=8.5, color="#1b5e20" if dtot>0 else "#b71c1c",
                fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#f1f8e9" if dtot>0 else "#ffebee",
                          edgecolor="#a5d6a7" if dtot>0 else "#ef9a9a"))

    # ── Col 1: Per-agent trajectory (skip convergence — it's image-level) ─────
    ax2 = axes[row_i, 1]
    ax2.set_facecolor("#fafafa")
    for sp in ax2.spines.values(): sp.set_color("#ddd")

    if metric != "convergence":
        for agent in debate_agents_list:
            ag_sub = sub[sub["agent"]==agent]
            ag_means = ag_sub.groupby("round")["score"].mean().reindex(ROUND_ORDER)
            ag_sems  = ag_sub.groupby("round")["score"].sem().reindex(ROUND_ORDER)
            col = AGENT_COLORS.get(agent, "#555")
            short_name = agent.replace("GPT-4.1-mini","GPT-mini")                                .replace("Claude-Sonnet","C-Sonnet")                                .replace("Claude-Haiku","C-Haiku")                                .replace("Grok-4.20-Fast","Grok-4.20")                                .replace("Gemma-3n-E4B","Gemma-3n")
            ax2.plot(x, ag_means.values, "o-", color=col, lw=2, ms=7,
                     label=short_name, zorder=3)
            ax2.fill_between(x,
                             ag_means.values - ag_sems.values*1.96,
                             ag_means.values + ag_sems.values*1.96,
                             alpha=0.12, color=col, zorder=2)
            # Label end point
            if not np.isnan(ag_means.values[-1]):
                ax2.text(2.05, ag_means.values[-1], f"{ag_means.values[-1]:.2f}",
                         va="center", fontsize=8, color=col, fontweight="bold")
        ax2.legend(fontsize=8.5, loc="lower left", framealpha=0.9)
        ax2.set_title("Per-Agent Trajectory", fontsize=11, fontweight="bold", pad=8)
    else:
        # Convergence: split by consensus outcome
        for ok, label, col in [(True,"Consensus Correct","#2e7d32"),
                               (False,"Consensus Incorrect","#c62828")]:
            grp = sub[sub["consensus_ok"]==ok]
            gm  = grp.groupby("round")["score"].mean().reindex(ROUND_ORDER)
            gs  = grp.groupby("round")["score"].sem().reindex(ROUND_ORDER)
            ax2.plot(x, gm.values, "o-", color=col, lw=2, ms=7, label=label, zorder=3)
            ax2.fill_between(x, gm.values-gs.values*1.96, gm.values+gs.values*1.96,
                             alpha=0.12, color=col, zorder=2)
        ax2.legend(fontsize=8.5, loc="lower left", framealpha=0.9)
        ax2.set_title("Convergence by Consensus Outcome", fontsize=11, fontweight="bold", pad=8)

    ax2.set_xticks(x); ax2.set_xticklabels([ROUND_LABELS[r] for r in ROUND_ORDER], fontsize=10)
    ax2.set_ylim(0, 11); ax2.set_ylabel(meta["ylabel"], fontsize=10)
    ax2.grid(axis="y", alpha=0.3, zorder=0)

    # ── Col 2: Score distribution violin per round ────────────────────────────
    ax3 = axes[row_i, 2]
    ax3.set_facecolor("#fafafa")
    for sp in ax3.spines.values(): sp.set_color("#ddd")

    violin_data = [sub[sub["round"]==r]["score"].dropna().values for r in ROUND_ORDER]
    vp = ax3.violinplot(violin_data, positions=x, widths=0.55,
                        showmedians=True, showextrema=False)
    for body, rnd in zip(vp["bodies"], ROUND_ORDER):
        body.set_facecolor(meta["color"]); body.set_alpha(0.4)
    vp["cmedians"].set_color("#111"); vp["cmedians"].set_linewidth(2)

    # Overlay mean dots
    for xi, rnd in zip(x, ROUND_ORDER):
        vals = sub[sub["round"]==rnd]["score"].dropna()
        ax3.scatter([xi]*len(vals), vals, alpha=0.2, s=8, color=meta["color"], zorder=2)
        ax3.scatter(xi, vals.mean(), s=60, color="#111", zorder=5, marker="D")

    ax3.set_xticks(x); ax3.set_xticklabels([ROUND_LABELS[r] for r in ROUND_ORDER], fontsize=10)
    ax3.set_ylim(-0.5, 11); ax3.set_ylabel("Score distribution", fontsize=10)
    ax3.set_title("Score Distribution Per Round", fontsize=11, fontweight="bold", pad=8)
    ax3.grid(axis="y", alpha=0.3, zorder=0)

plt.tight_layout(h_pad=3.5, w_pad=2.5)
plt.savefig(OUTPUT_DIR/"semantic_eval_visualization.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

# ── Print summary table ───────────────────────────────────────────────────────
print("\n" + "="*72)
print("  SEMANTIC EVALUATION SUMMARY")
print("="*72)
summary = sem_df.dropna(subset=["score"]).groupby(["metric","round"])["score"].agg(["mean","sem","count"])
summary["95ci"] = summary["sem"] * 1.96
for metric in ["label_similarity","specificity","convergence"]:
    print(f"\n  {METRIC_META[metric]['title']}")
    print(f"  {'Round':<8} {'Mean':>8} {'95% CI':>10} {'N':>6}")
    print("  "+"-"*36)
    for rnd in ROUND_ORDER:
        try:
            row = summary.loc[(metric,rnd)]
            print(f"  {rnd:<8} {row['mean']:>8.3f} {row['95ci']:>10.3f} {int(row['count']):>6}")
        except KeyError:
            print(f"  {rnd:<8} {'N/A':>8}")
    # Delta R1→R3
    try:
        d = summary.loc[(metric,"r3")]["mean"] - summary.loc[(metric,"r1")]["mean"]
        dir_str = "IMPROVED" if d>0.15 else ("DECLINED" if d<-0.15 else "STABLE")
        print(f"  Delta R1→R3: {d:+.3f}  [{dir_str}]")
    except: pass

print("\n[OK] Visualization saved: semantic_eval_visualization.png")
